# Čišćenje i priprema Olist Brazilian E-Commerce podataka

Ova bilježnica predstavlja drugu fazu praktičnog dijela rada i nastavlja se na rezultate početne eksplorativne analize provedene u bilježnici `01_analiza_podataka.ipynb`.

Cilj ove faze je pripremiti podatke za kasnije projektiranje i izgradnju skladišta podataka. Čišćenje se provodi na temelju problema kvalitete podataka, anomalija i poslovnih pravila utvrđenih tijekom prethodne analize.

U ovoj bilježnici provest će se:

- učitavanje izvornih CSV datoteka,
- izrada radnih kopija izvornih skupova podataka,
- standardizacija tipova podataka,
- uklanjanje samo potvrđenih potpunih duplikata,
- standardizacija tekstualnih vrijednosti gdje je to opravdano,
- obrada nedostajućih vrijednosti prema poslovnom značenju,
- dodavanje nedostajućih prijevoda kategorija proizvoda,
- označavanje uočenih anomalija bez automatskog uklanjanja potencijalno valjanih poslovnih zapisa,
- provjera kvalitete podataka nakon provedenih transformacija,
- spremanje očišćenih skupova podataka za daljnju obradu.

Izvorne CSV datoteke neće se mijenjati. Sve transformacije provodit će se nad zasebnim radnim kopijama podataka.

In [507]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

print("Pandas verzija:", pd.__version__)
print("NumPy verzija:", np.__version__)
print("Trenutni radni direktorij:", os.getcwd())

Pandas verzija: 3.0.3
NumPy verzija: 2.3.2
Trenutni radni direktorij: c:\Users\Ana\Desktop\diplomskiPrakticni\notebooks


In [508]:
# Korijenski direktorij projekta nalazi se jednu razinu iznad mape notebooks
PROJECT_DIR = Path("..").resolve()

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
CLEAN_DATA_DIR = PROJECT_DIR / "data" / "clean"

# Izrada direktorija ako ne postoji
CLEAN_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Korijenski direktorij projekta:")
print(PROJECT_DIR)

print("\nDirektorij izvornih podataka:")
print(RAW_DATA_DIR)

print("\nDirektorij očišćenih podataka:")
print(CLEAN_DATA_DIR)

Korijenski direktorij projekta:
C:\Users\Ana\Desktop\diplomskiPrakticni

Direktorij izvornih podataka:
C:\Users\Ana\Desktop\diplomskiPrakticni\data\raw

Direktorij očišćenih podataka:
C:\Users\Ana\Desktop\diplomskiPrakticni\data\clean


## Učitavanje izvornih podataka

Učitava se svih devet izvornih CSV datoteka Olist skupa podataka.

U ovoj fazi podaci se učitavaju bez promjene sadržaja. Nakon učitavanja izradit će se zasebne radne kopije nad kojima će se provoditi čišćenje i transformacije.

In [509]:
customers_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_customers_dataset.csv"
)

geolocation_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_geolocation_dataset.csv"
)

orders_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_orders_dataset.csv"
)

order_items_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_order_items_dataset.csv"
)

payments_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_order_payments_dataset.csv"
)

reviews_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_order_reviews_dataset.csv"
)

products_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_products_dataset.csv"
)

sellers_raw = pd.read_csv(
    RAW_DATA_DIR / "olist_sellers_dataset.csv"
)

category_translation_raw = pd.read_csv(
    RAW_DATA_DIR / "product_category_name_translation.csv"
)

## Izrada radnih kopija

Čišćenje se neće provoditi izravno nad izvornim DataFrameovima. Za svaki izvorni skup izrađuje se kopija s nastavkom `_clean`.

Time se omogućuje:

- očuvanje izvornih podataka,
- usporedba stanja prije i nakon čišćenja,
- lakša provjera utjecaja pojedinih transformacija,
- ponovno izvođenje postupka bez ponovnog učitavanja datoteka.

In [510]:
customers_clean = customers_raw.copy(deep=True)
geolocation_clean = geolocation_raw.copy(deep=True)
orders_clean = orders_raw.copy(deep=True)
order_items_clean = order_items_raw.copy(deep=True)
payments_clean = payments_raw.copy(deep=True)
reviews_clean = reviews_raw.copy(deep=True)
products_clean = products_raw.copy(deep=True)
sellers_clean = sellers_raw.copy(deep=True)
category_translation_clean = category_translation_raw.copy(deep=True)

## Početni pregled učitanih skupova podataka

Prije čišćenja provjerava se jesu li svi skupovi uspješno učitani te imaju li radne kopije jednak broj redaka i stupaca kao izvorni podaci.

In [511]:
raw_dataframes = {
    "customers": customers_raw,
    "geolocation": geolocation_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
    "payments": payments_raw,
    "reviews": reviews_raw,
    "products": products_raw,
    "sellers": sellers_raw,
    "category_translation": category_translation_raw
}

clean_dataframes = {
    "customers": customers_clean,
    "geolocation": geolocation_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean,
    "category_translation": category_translation_clean
}

pocetni_pregled = pd.DataFrame(
    {
        "dataset": raw_dataframes.keys(),
        "raw_broj_redaka": [
            df.shape[0]
            for df in raw_dataframes.values()
        ],
        "clean_broj_redaka": [
            df.shape[0]
            for df in clean_dataframes.values()
        ],
        "raw_broj_stupaca": [
            df.shape[1]
            for df in raw_dataframes.values()
        ],
        "clean_broj_stupaca": [
            df.shape[1]
            for df in clean_dataframes.values()
        ]
    }
)

pocetni_pregled

,dataset,raw_broj_redaka,clean_broj_redaka,raw_broj_stupaca,clean_broj_stupaca
0,customers,99441,99441,5,5
1,geolocation,1000163,1000163,5,5
2,orders,99441,99441,8,8
3,order_items,112650,112650,7,7
4,payments,103886,103886,5,5
5,reviews,99224,99224,7,7
6,products,32951,32951,9,9
7,sellers,3095,3095,4,4
8,category_translation,71,71,2,2


In [512]:
pocetni_pregled["jednake_dimenzije"] = (
    (
        pocetni_pregled["raw_broj_redaka"]
        == pocetni_pregled["clean_broj_redaka"]
    )
    &
    (
        pocetni_pregled["raw_broj_stupaca"]
        == pocetni_pregled["clean_broj_stupaca"]
    )
)

print(
    "Svi izvorni i radni DataFrameovi imaju jednake dimenzije:",
    pocetni_pregled["jednake_dimenzije"].all()
)

pocetni_pregled

Svi izvorni i radni DataFrameovi imaju jednake dimenzije: True


,dataset,raw_broj_redaka,clean_broj_redaka,raw_broj_stupaca,clean_broj_stupaca,jednake_dimenzije
0,customers,99441,99441,5,5,True
1,geolocation,1000163,1000163,5,5,True
2,orders,99441,99441,8,8,True
3,order_items,112650,112650,7,7,True
4,payments,103886,103886,5,5,True
5,reviews,99224,99224,7,7,True
6,products,32951,32951,9,9,True
7,sellers,3095,3095,4,4,True
8,category_translation,71,71,2,2,True


### Zaključak početne pripreme

Svi izvorni skupovi podataka uspješno su učitani, a za svaki od njih izrađena je zasebna radna kopija.

U ovom trenutku radne kopije imaju jednak broj redaka i stupaca kao izvorni podaci. Sve daljnje promjene provodit će se isključivo nad DataFrameovima s nastavkom `_clean`, dok će DataFrameovi s nastavkom `_raw` ostati nepromijenjeni i služiti kao referentno stanje.

## Početni izvještaj o kvaliteti podataka

Prije provedbe pojedinačnih transformacija izrađuje se početni izvještaj o kvaliteti podataka za svaki učitani skup.

Izvještaj za svaki stupac prikazuje:

- tip podatka,
- broj i udio nedostajućih vrijednosti,
- broj jedinstvenih vrijednosti,
- broj dupliciranih vrijednosti unutar stupca.

Na razini cijele tablice dodatno se bilježi broj redaka, broj stupaca i broj potpuno dupliciranih redaka.

Ista provjera ponovit će se nakon čišćenja kako bi se mogao jasno prikazati učinak provedenih transformacija.

In [513]:
def izvjestaj_kvalitete(df):
    """
    Vraća izvještaj o kvaliteti podataka na razini stupaca.
    """
    izvjestaj = pd.DataFrame({
        "tip_podatka": df.dtypes.astype(str),
        "broj_nedostajucih": df.isna().sum(),
        "udio_nedostajucih_pct": (
            df.isna().mean().mul(100).round(2)
        ),
        "broj_jedinstvenih": df.nunique(dropna=True),
        "broj_dupliciranih_vrijednosti": (
            df.shape[0] - df.nunique(dropna=False)
        )
    })

    return izvjestaj


def sazetak_tablice(df):
    """
    Vraća osnovni sažetak kvalitete cijele tablice.
    """
    return {
        "broj_redaka": df.shape[0],
        "broj_stupaca": df.shape[1],
        "broj_potpuno_dupliciranih_redaka": df.duplicated().sum(),
        "ukupan_broj_nedostajucih_vrijednosti": (
            df.isna().sum().sum()
        )
    }

In [514]:
pocetni_sazeci = []

for naziv, df in raw_dataframes.items():
    sazetak = sazetak_tablice(df)

    pocetni_sazeci.append({
        "dataset": naziv,
        **sazetak
    })

pocetni_sazetak_kvalitete = pd.DataFrame(
    pocetni_sazeci
)

pocetni_sazetak_kvalitete

,dataset,broj_redaka,broj_stupaca,broj_potpuno_dupliciranih_redaka,ukupan_broj_nedostajucih_vrijednosti
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,orders,99441,8,0,4908
3,order_items,112650,7,0,0
4,payments,103886,5,0,0
5,reviews,99224,7,0,145903
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


### Početno stanje kvalitete podataka

Početni izvještaj služi kao referentno stanje prije čišćenja. Pojava nedostajućih vrijednosti, ponovljenih identifikatora ili potpuno dupliciranih redaka ne znači automatski da se podaci trebaju ukloniti.

Svaka potencijalna nepravilnost analizirat će se zasebno, uzimajući u obzir značenje atributa i poslovni kontekst tablice.

# Pomoćne funkcije za čišćenje podataka

Prije čišćenja pojedinačnih tablica definiraju se pomoćne funkcije koje će se koristiti tijekom cijelog procesa.

Nazivi gradova u izvornim tablicama nisu uvijek zapisani na isti način. Razlike uključuju velika i mala slova, dijakritičke znakove, interpunkciju, posebne znakove i višestruke razmake.

Standardizacijom se uklanjaju tehničke razlike u zapisu, ali se ne pokušavaju automatski ispravljati tipografske ili semantičke pogreške. Takve će se vrijednosti ispravljati samo kada ih je moguće dovoljno pouzdano potvrditi pomoću ZIP prefiksa, grada i savezne države.

In [515]:
import re
import unicodedata


def standardiziraj_tekst(vrijednost):
    """
    Standardizira tekstualnu vrijednost:

    - pretvara tekst u mala slova
    - uklanja dijakritičke znakove
    - posebne znakove zamjenjuje razmakom
    - višestruke razmake svodi na jedan

    Nedostajuće vrijednosti ostaju nedostajuće.
    """
    if pd.isna(vrijednost):
        return pd.NA

    tekst = str(vrijednost).strip().lower()

    tekst = (
        unicodedata.normalize("NFKD", tekst)
        .encode("ascii", "ignore")
        .decode("utf-8")
    )

    tekst = re.sub(r"[^a-z0-9]+", " ", tekst)
    tekst = re.sub(r"\s+", " ", tekst).strip()

    return tekst

In [516]:
primjeri_teksta = pd.Series([
    "São Paulo",
    "  SAO   PAULO  ",
    "sao paulo / sao paulo",
    "Goiô-Erê",
    pd.NA
])

pd.DataFrame({
    "izvorna_vrijednost": primjeri_teksta,
    "standardizirana_vrijednost": (
        primjeri_teksta.apply(standardiziraj_tekst)
    )
})

,izvorna_vrijednost,standardizirana_vrijednost
0,São Paulo,sao paulo
1,SAO PAULO,sao paulo
2,sao paulo / sao paulo,sao paulo sao paulo
3,Goiô-Erê,goio ere
4,NaN,NaN


# 1. Čišćenje tablice `geolocation`

Tablica `geolocation` služi kao referentni izvor za provjeru geografskih podataka kupaca i prodavača.

Čišćenje se provodi konzervativno:

- uklanjaju se samo potpuno identični retci;
- izvorni grad i savezna država ostaju sačuvani;
- stvaraju se standardizirani i očišćeni atributi;
- svako pravilo ispravke prvo se validira nad podacima iz tablice `geolocation`;
- višeznačne veze koje mogu predstavljati različita naselja ili administrativne cjeline ne ispravljaju se automatski.

In [517]:
print("Broj redaka prije čišćenja:", len(geolocation_clean))
print("Broj potpuno dupliciranih redaka:", geolocation_clean.duplicated().sum())
print("\nNedostajuće vrijednosti:")
print(geolocation_clean.isna().sum())

Broj redaka prije čišćenja: 1000163
Broj potpuno dupliciranih redaka: 261831

Nedostajuće vrijednosti:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64


## Uklanjanje potpuno dupliciranih geografskih zapisa

Potpuno identični retci ne donose novu informaciju te se uklanjaju. Zapisi s istim ZIP prefiksom, ali različitim gradom, državom ili koordinatama ne smatraju se automatski duplikatima.

In [518]:
broj_redaka_geo_prije = len(geolocation_clean)

geolocation_clean = (
    geolocation_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

broj_redaka_geo_poslije = len(geolocation_clean)

print("Broj redaka prije:", broj_redaka_geo_prije)
print("Broj redaka poslije:", broj_redaka_geo_poslije)
print(
    "Broj uklonjenih potpuno dupliciranih redaka:",
    broj_redaka_geo_prije - broj_redaka_geo_poslije
)

Broj redaka prije: 1000163
Broj redaka poslije: 738332
Broj uklonjenih potpuno dupliciranih redaka: 261831


## Standardizacija grada i savezne države

Nazivi gradova standardiziraju se uklanjanjem tehničkih razlika u zapisu. Izvorni atributi ostaju nepromijenjeni.

In [519]:
geolocation_clean["geolocation_city_std"] = (
    geolocation_clean["geolocation_city"]
    .apply(standardiziraj_tekst)
)

geolocation_clean["geolocation_state_std"] = (
    geolocation_clean["geolocation_state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

geolocation_clean["geolocation_city_clean"] = (
    geolocation_clean["geolocation_city_std"].copy()
)

geolocation_clean["geolocation_state_clean"] = (
    geolocation_clean["geolocation_state_std"].copy()
)

geolocation_clean["geolocation_city_corrected"] = False
geolocation_clean["geolocation_state_corrected"] = False

geolocation_clean["geolocation_correction_method"] = pd.Series(
    "nije mijenjano",
    index=geolocation_clean.index,
    dtype="string"
)

In [520]:
print(
    "Broj jedinstvenih izvornih gradova:",
    geolocation_clean["geolocation_city"].nunique()
)
print(
    "Broj jedinstvenih standardiziranih gradova:",
    geolocation_clean["geolocation_city_std"].nunique()
)
print(
    "Broj jedinstvenih saveznih država:",
    geolocation_clean["geolocation_state_std"].nunique()
)

Broj jedinstvenih izvornih gradova: 8011
Broj jedinstvenih standardiziranih gradova: 5929
Broj jedinstvenih saveznih država: 27


## Validacija i ispravak očitih pogrešaka saveznih država

Pet prethodno uočenih pravila ne primjenjuju se neposredno. Za svako se pravilo provjerava učestalost svih saveznih država uz isti ZIP prefiks i grad.

Pravilo se potvrđuje samo kada:

- pogrešna vrijednost stvarno postoji;
- predložena država pojavljuje se češće;
- predložena država najmanje je pet puta učestalija;
- nema treće konkurentske savezne države.

In [521]:
predlozene_ispravke_drzava_geo = pd.DataFrame(
    [
        {
            "zip": 79750,
            "grad": "nova andradina",
            "pogresna_drzava": "RS",
            "ispravna_drzava": "MS"
        },
        {
            "zip": 21550,
            "grad": "rio de janeiro",
            "pogresna_drzava": "AC",
            "ispravna_drzava": "RJ"
        },
        {
            "zip": 23056,
            "grad": "rio de janeiro",
            "pogresna_drzava": "AC",
            "ispravna_drzava": "RJ"
        },
        {
            "zip": 4011,
            "grad": "sao paulo",
            "pogresna_drzava": "AC",
            "ispravna_drzava": "SP"
        },
        {
            "zip": 2116,
            "grad": "sao paulo",
            "pogresna_drzava": "RN",
            "ispravna_drzava": "SP"
        }
    ]
)

predlozene_ispravke_drzava_geo

,zip,grad,pogresna_drzava,ispravna_drzava
0,79750,nova andradina,RS,MS
1,21550,rio de janeiro,AC,RJ
2,23056,rio de janeiro,AC,RJ
3,4011,sao paulo,AC,SP
4,2116,sao paulo,RN,SP


In [522]:
validacija_ispravaka_drzava_geo = []

for _, pravilo in predlozene_ispravke_drzava_geo.iterrows():
    ucestalosti = (
        geolocation_clean.loc[
            (
                geolocation_clean["geolocation_zip_code_prefix"]
                == pravilo["zip"]
            )
            &
            (
                geolocation_clean["geolocation_city_clean"]
                == pravilo["grad"]
            ),
            "geolocation_state_clean"
        ]
        .value_counts()
    )

    broj_pogresne = int(
        ucestalosti.get(pravilo["pogresna_drzava"], 0)
    )
    broj_ispravne = int(
        ucestalosti.get(pravilo["ispravna_drzava"], 0)
    )

    ostale_drzave = [
        drzava
        for drzava in ucestalosti.index
        if drzava not in {
            pravilo["pogresna_drzava"],
            pravilo["ispravna_drzava"]
        }
    ]

    potvrdeno = (
        broj_pogresne > 0
        and broj_ispravne > broj_pogresne
        and broj_ispravne >= 5 * broj_pogresne
        and len(ostale_drzave) == 0
    )

    validacija_ispravaka_drzava_geo.append(
        {
            **pravilo.to_dict(),
            "broj_pogresne_drzave": broj_pogresne,
            "broj_ispravne_drzave": broj_ispravne,
            "ostale_drzave": ostale_drzave,
            "potvrdeno": potvrdeno
        }
    )

validacija_ispravaka_drzava_geo = pd.DataFrame(
    validacija_ispravaka_drzava_geo
)

validacija_ispravaka_drzava_geo

,zip,grad,pogresna_drzava,ispravna_drzava,broj_pogresne_drzave,broj_ispravne_drzave,ostale_drzave,potvrdeno
0,79750,nova andradina,RS,MS,1,151,[],True
1,21550,rio de janeiro,AC,RJ,1,144,[],True
2,23056,rio de janeiro,AC,RJ,1,30,[],True
3,4011,sao paulo,AC,SP,1,69,[],True
4,2116,sao paulo,RN,SP,1,10,[],True


In [523]:
potvrdene_ispravke_drzava_geo = (
    validacija_ispravaka_drzava_geo[
        validacija_ispravaka_drzava_geo["potvrdeno"]
    ]
    .copy()
)

print(
    "Broj potvrđenih pravila za ispravak države:",
    len(potvrdene_ispravke_drzava_geo)
)

assert (
    len(potvrdene_ispravke_drzava_geo)
    == len(predlozene_ispravke_drzava_geo)
), (
    "Nisu sva predložena pravila potvrđena. "
    "Pregledaj validacijsku tablicu prije primjene."
)

Broj potvrđenih pravila za ispravak države: 5


In [524]:
for _, ispravka in potvrdene_ispravke_drzava_geo.iterrows():
    maska = (
        (
            geolocation_clean["geolocation_zip_code_prefix"]
            == ispravka["zip"]
        )
        &
        (
            geolocation_clean["geolocation_city_clean"]
            == ispravka["grad"]
        )
        &
        (
            geolocation_clean["geolocation_state_clean"]
            == ispravka["pogresna_drzava"]
        )
    )

    geolocation_clean.loc[
        maska,
        "geolocation_state_clean"
    ] = ispravka["ispravna_drzava"]

    geolocation_clean.loc[
        maska,
        "geolocation_state_corrected"
    ] = True

    geolocation_clean.loc[
        maska,
        "geolocation_correction_method"
    ] = "ispravak države validiran učestalošću u geolocation"

## Pomoćne funkcije za usporedbu gradova

In [525]:
def kompaktni_naziv(vrijednost):
    if pd.isna(vrijednost):
        return pd.NA

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(vrijednost).lower()
    )


def levenshtein_udaljenost(tekst_1, tekst_2):
    if pd.isna(tekst_1) or pd.isna(tekst_2):
        return pd.NA

    tekst_1 = str(tekst_1)
    tekst_2 = str(tekst_2)

    prethodni_redak = list(range(len(tekst_2) + 1))

    for i, znak_1 in enumerate(tekst_1, start=1):
        trenutni_redak = [i]

        for j, znak_2 in enumerate(tekst_2, start=1):
            cijena_zamjene = 0 if znak_1 == znak_2 else 1

            trenutni_redak.append(
                min(
                    prethodni_redak[j] + 1,
                    trenutni_redak[j - 1] + 1,
                    prethodni_redak[j - 1] + cijena_zamjene
                )
            )

        prethodni_redak = trenutni_redak

    return prethodni_redak[-1]

## Standardizacija potvrđenih inačica naziva São Paulo

Inačica se mijenja u `sao paulo` samo kada:

- pripada saveznoj državi `SP`;
- isti ZIP prefiks već sadrži potvrđeni zapis `sao paulo`;
- zapis je poznata inačica, jednak je nakon uklanjanja znakova ili se razlikuje u najviše dva znaka.

In [526]:
poznate_varijante_sao_paulo = {
    "sao paulo",
    "saopaulo",
    "sao paolo",
    "sao pauo",
    "sao paulop",
    "sao paullo",
    "sao paoulo",
    "sao paulo sp",
    "sao paulo sao paulo",
    "s paulo",
    "sp sp"
}

zipovi_s_potvrdenim_sao_paulo = set(
    geolocation_clean.loc[
        (
            geolocation_clean["geolocation_state_clean"] == "SP"
        )
        &
        (
            geolocation_clean["geolocation_city_clean"] == "sao paulo"
        ),
        "geolocation_zip_code_prefix"
    ]
)

kandidati_sao_paulo = geolocation_clean[
    (
        geolocation_clean["geolocation_state_clean"] == "SP"
    )
    &
    (
        geolocation_clean["geolocation_zip_code_prefix"].isin(
            zipovi_s_potvrdenim_sao_paulo
        )
    )
].copy()

kandidati_sao_paulo["kompaktni_grad"] = (
    kandidati_sao_paulo["geolocation_city_clean"]
    .apply(kompaktni_naziv)
)

kandidati_sao_paulo["udaljenost_od_sao_paulo"] = (
    kandidati_sao_paulo["kompaktni_grad"]
    .apply(
        lambda naziv: (
            levenshtein_udaljenost(naziv, "saopaulo")
            if pd.notna(naziv)
            else pd.NA
        )
    )
)

maska_sao_paulo = (
    (
        kandidati_sao_paulo["geolocation_city_clean"].isin(
            poznate_varijante_sao_paulo
        )
    )
    |
    (
        kandidati_sao_paulo["kompaktni_grad"] == "saopaulo"
    )
    |
    (
        kandidati_sao_paulo["udaljenost_od_sao_paulo"].le(2)
        &
        kandidati_sao_paulo[
            "geolocation_city_clean"
        ].str.split().str.len().le(3)
    )
)

varijante_sao_paulo_za_ispravak = (
    kandidati_sao_paulo[
        maska_sao_paulo
        &
        (
            kandidati_sao_paulo["geolocation_city_clean"]
            != "sao paulo"
        )
    ]
    .copy()
)

assert (
    varijante_sao_paulo_za_ispravak[
        "geolocation_state_clean"
    ].eq("SP").all()
)

assert (
    varijante_sao_paulo_za_ispravak[
        "geolocation_zip_code_prefix"
    ].isin(zipovi_s_potvrdenim_sao_paulo).all()
)

print(
    "Broj jedinstvenih potvrđenih inačica São Paula:",
    varijante_sao_paulo_za_ispravak[
        "geolocation_city_clean"
    ].nunique()
)

varijante_sao_paulo_za_ispravak[
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_city_clean",
        "geolocation_state_clean",
        "udaljenost_od_sao_paulo"
    ]
].drop_duplicates()

Broj jedinstvenih potvrđenih inačica São Paula: 1


,geolocation_zip_code_prefix,geolocation_city,geolocation_city_clean,geolocation_state_clean,udaljenost_od_sao_paulo
49274,4132,sãopaulo,saopaulo,SP,0
125100,8120,sãopaulo,saopaulo,SP,0


In [527]:
varijante_sao_paulo = set(
    varijante_sao_paulo_za_ispravak[
        "geolocation_city_clean"
    ]
)

maska_ispravka_sao_paulo = (
    (
        geolocation_clean["geolocation_state_clean"] == "SP"
    )
    &
    (
        geolocation_clean["geolocation_zip_code_prefix"].isin(
            zipovi_s_potvrdenim_sao_paulo
        )
    )
    &
    (
        geolocation_clean["geolocation_city_clean"].isin(
            varijante_sao_paulo
        )
    )
)

broj_ispravaka_sao_paulo = int(
    maska_ispravka_sao_paulo.sum()
)

geolocation_clean.loc[
    maska_ispravka_sao_paulo,
    "geolocation_city_clean"
] = "sao paulo"

geolocation_clean.loc[
    maska_ispravka_sao_paulo,
    "geolocation_city_corrected"
] = True

geolocation_clean.loc[
    maska_ispravka_sao_paulo,
    "geolocation_correction_method"
] = (
    "inačica São Paula potvrđena ZIP-om i državom u geolocation"
)

print(
    "Broj zapisa standardiziranih u 'sao paulo':",
    broj_ispravaka_sao_paulo
)

Broj zapisa standardiziranih u 'sao paulo': 2


## Konzervativna validacija ostalih tipografskih inačica

Automatska ispravka dopuštena je samo unutar iste kombinacije ZIP prefiksa i savezne države, uz dominantan ispravan naziv i vrlo malu tekstualnu razliku.

In [528]:
from itertools import combinations
from difflib import SequenceMatcher

ucestalost_geo_gradova = (
    geolocation_clean
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_state_clean",
            "geolocation_city_clean"
        ]
    )
    .size()
    .reset_index(name="broj_pojavljivanja")
)

parovi_geo_gradova = []

for (zip_prefiks, drzava), grupa in ucestalost_geo_gradova.groupby(
    [
        "geolocation_zip_code_prefix",
        "geolocation_state_clean"
    ]
):
    if grupa["geolocation_city_clean"].nunique() <= 1:
        continue

    zapisi = grupa[
        [
            "geolocation_city_clean",
            "broj_pojavljivanja"
        ]
    ].to_records(index=False)

    for zapis_1, zapis_2 in combinations(zapisi, 2):
        grad_1, broj_1 = zapis_1[0], int(zapis_1[1])
        grad_2, broj_2 = zapis_2[0], int(zapis_2[1])

        if broj_1 >= broj_2:
            ispravan_grad, broj_ispravan = grad_1, broj_1
            pogresan_grad, broj_pogresan = grad_2, broj_2
        else:
            ispravan_grad, broj_ispravan = grad_2, broj_2
            pogresan_grad, broj_pogresan = grad_1, broj_1

        udaljenost = levenshtein_udaljenost(
            kompaktni_naziv(grad_1),
            kompaktni_naziv(grad_2)
        )

        slicnost = round(
            SequenceMatcher(
                None,
                str(grad_1),
                str(grad_2)
            ).ratio() * 100,
            2
        )

        parovi_geo_gradova.append(
            {
                "geolocation_zip_code_prefix": zip_prefiks,
                "geolocation_state_clean": drzava,
                "pogresan_grad": pogresan_grad,
                "ispravan_grad": ispravan_grad,
                "broj_pogresan": broj_pogresan,
                "broj_ispravan": broj_ispravan,
                "omjer_ucestalosti": (
                    broj_ispravan / broj_pogresan
                    if broj_pogresan > 0
                    else float("inf")
                ),
                "levenshtein_udaljenost": udaljenost,
                "slicnost_pct": slicnost,
                "isti_kompaktni_naziv": (
                    kompaktni_naziv(grad_1)
                    == kompaktni_naziv(grad_2)
                )
            }
        )

kandidati_geo_tipfeleri = pd.DataFrame(parovi_geo_gradova)

sigurne_ispravke_geo_gradova = (
    kandidati_geo_tipfeleri[
        (
            kandidati_geo_tipfeleri["broj_pogresan"] <= 2
        )
        &
        (
            kandidati_geo_tipfeleri["omjer_ucestalosti"] >= 5
        )
        &
        (
            kandidati_geo_tipfeleri["isti_kompaktni_naziv"]
            |
            (
                kandidati_geo_tipfeleri[
                    "levenshtein_udaljenost"
                ] <= 1
            )
        )
        &
        (
            kandidati_geo_tipfeleri["slicnost_pct"] >= 85
        )
    ]
    .drop_duplicates(
        subset=[
            "geolocation_zip_code_prefix",
            "geolocation_state_clean",
            "pogresan_grad"
        ]
    )
    .copy()
)

print(
    "Broj validiranih pravila za ostale gradove:",
    len(sigurne_ispravke_geo_gradova)
)

sigurne_ispravke_geo_gradova

Broj validiranih pravila za ostale gradove: 32


,geolocation_zip_code_prefix,geolocation_state_clean,pogresan_grad,ispravan_grad,broj_pogresan,broj_ispravan,omjer_ucestalosti,levenshtein_udaljenost,slicnost_pct,isti_kompaktni_naziv
43,8730,SP,mogidascruzes,mogi das cruzes,1,176,176.0,0,92.86,True
61,13450,SP,santa barbara doeste,santa barbara d oeste,1,137,137.0,0,97.56,True
62,13453,SP,santa barbara doeste,santa barbara d oeste,1,70,70.0,0,97.56,True
65,13456,SP,santa barbara doeste,santa barbara d oeste,2,80,40.0,0,97.56,True
74,15720,SP,palmeira doeste,palmeira d oeste,1,36,36.0,0,96.77,True
75,15735,SP,aparecida doeste,aparecida d oeste,1,20,20.0,0,96.97,True
76,15780,SP,santa rita doeste,santa rita d oeste,1,10,10.0,0,97.14,True
83,17970,SP,sao joao do pau dalho,sao joao do pau d alho,2,12,6.0,0,97.67,True
135,28026,RJ,campos dos goytacaze,campos dos goytacazes,1,69,69.0,1,97.56,False
168,29665,ES,sao roque do cannaa,sao roque do canaa,2,20,10.0,1,97.30,False


In [529]:
provjera_sigurnih_geo_ispravaka = (
    geolocation_clean
    .merge(
        sigurne_ispravke_geo_gradova[
            [
                "geolocation_zip_code_prefix",
                "geolocation_state_clean",
                "pogresan_grad",
                "ispravan_grad"
            ]
        ],
        left_on=[
            "geolocation_zip_code_prefix",
            "geolocation_state_clean",
            "geolocation_city_clean"
        ],
        right_on=[
            "geolocation_zip_code_prefix",
            "geolocation_state_clean",
            "pogresan_grad"
        ],
        how="inner"
    )
)

print(
    "Broj zapisa obuhvaćenih validiranim pravilima:",
    len(provjera_sigurnih_geo_ispravaka)
)

provjera_sigurnih_geo_ispravaka[
    [
        "geolocation_zip_code_prefix",
        "geolocation_state_clean",
        "geolocation_city",
        "geolocation_city_clean",
        "ispravan_grad"
    ]
].drop_duplicates()

Broj zapisa obuhvaćenih validiranim pravilima: 39


,geolocation_zip_code_prefix,geolocation_state_clean,geolocation_city,geolocation_city_clean,ispravan_grad
0,8730,SP,mogidascruzes,mogidascruzes,mogi das cruzes
1,13456,SP,santa bárbara doeste,santa barbara doeste,santa barbara d oeste
2,13450,SP,santa bárbara doeste,santa barbara doeste,santa barbara d oeste
4,13453,SP,santa barbara doeste,santa barbara doeste,santa barbara d oeste
5,15720,SP,palmeira doeste,palmeira doeste,palmeira d oeste
6,15735,SP,aparecida doeste,aparecida doeste,aparecida d oeste
7,15780,SP,santa rita doeste,santa rita doeste,santa rita d oeste
8,17970,SP,sao joao do pau dalho,sao joao do pau dalho,sao joao do pau d alho
10,28026,RJ,campos dos goytacaze,campos dos goytacaze,campos dos goytacazes
11,29665,ES,sao roque do cannaa,sao roque do cannaa,sao roque do canaa


In [530]:
for _, ispravka in sigurne_ispravke_geo_gradova.iterrows():
    maska = (
        (
            geolocation_clean["geolocation_zip_code_prefix"]
            == ispravka["geolocation_zip_code_prefix"]
        )
        &
        (
            geolocation_clean["geolocation_state_clean"]
            == ispravka["geolocation_state_clean"]
        )
        &
        (
            geolocation_clean["geolocation_city_clean"]
            == ispravka["pogresan_grad"]
        )
    )

    geolocation_clean.loc[
        maska,
        "geolocation_city_clean"
    ] = ispravka["ispravan_grad"]

    geolocation_clean.loc[
        maska,
        "geolocation_city_corrected"
    ] = True

    geolocation_clean.loc[
        maska,
        "geolocation_correction_method"
    ] = (
        "tipografska inačica validirana unutar geolocation"
    )

## Završna provjera i referentne tablice

In [531]:
provjera_vise_drzava_nakon = (
    geolocation_clean
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_city_clean"
        ]
    )
    .agg(
        broj_drzava=(
            "geolocation_state_clean",
            "nunique"
        ),
        drzave=(
            "geolocation_state_clean",
            lambda x: sorted(set(x.dropna()))
        )
    )
    .reset_index()
)

provjera_vise_drzava_nakon = (
    provjera_vise_drzava_nakon[
        provjera_vise_drzava_nakon["broj_drzava"] > 1
    ]
)

provjera_vise_gradova_nakon = (
    geolocation_clean
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_state_clean"
        ]
    )
    .agg(
        broj_gradova=(
            "geolocation_city_clean",
            "nunique"
        ),
        gradovi=(
            "geolocation_city_clean",
            lambda x: sorted(set(x.dropna()))
        )
    )
    .reset_index()
)

provjera_vise_gradova_nakon = (
    provjera_vise_gradova_nakon[
        provjera_vise_gradova_nakon["broj_gradova"] > 1
    ]
)

print(
    "ZIP + grad kombinacije s više država:",
    len(provjera_vise_drzava_nakon)
)
print(
    "ZIP + država kombinacije s više gradova:",
    len(provjera_vise_gradova_nakon)
)
print(
    "Zapisi s ispravljenim gradom:",
    int(geolocation_clean["geolocation_city_corrected"].sum())
)
print(
    "Zapisi s ispravljenom državom:",
    int(geolocation_clean["geolocation_state_corrected"].sum())
)

ZIP + grad kombinacije s više država: 0
ZIP + država kombinacije s više gradova: 480
Zapisi s ispravljenim gradom: 41
Zapisi s ispravljenom državom: 5


In [532]:
geo_zip_grad_konacno = (
    geolocation_clean
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_city_clean"
        ]
    )
    .agg(
        broj_drzava=(
            "geolocation_state_clean",
            "nunique"
        ),
        referentna_drzava=(
            "geolocation_state_clean",
            "first"
        )
    )
    .reset_index()
)

geo_ref_zip_grad = (
    geo_zip_grad_konacno[
        geo_zip_grad_konacno["broj_drzava"] == 1
    ][
        [
            "geolocation_zip_code_prefix",
            "geolocation_city_clean",
            "referentna_drzava"
        ]
    ]
    .copy()
)

geo_zip_drzava_konacno = (
    geolocation_clean
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_state_clean"
        ]
    )
    .agg(
        broj_gradova=(
            "geolocation_city_clean",
            "nunique"
        ),
        referentni_grad=(
            "geolocation_city_clean",
            "first"
        )
    )
    .reset_index()
)

geo_ref_zip_drzava = (
    geo_zip_drzava_konacno[
        geo_zip_drzava_konacno["broj_gradova"] == 1
    ][
        [
            "geolocation_zip_code_prefix",
            "geolocation_state_clean",
            "referentni_grad"
        ]
    ]
    .copy()
)

assert (
    geo_ref_zip_grad.duplicated(
        subset=[
            "geolocation_zip_code_prefix",
            "geolocation_city_clean"
        ]
    ).sum() == 0
)

assert (
    geo_ref_zip_drzava.duplicated(
        subset=[
            "geolocation_zip_code_prefix",
            "geolocation_state_clean"
        ]
    ).sum() == 0
)

print(
    "Jednoznačne ZIP + grad → država veze:",
    len(geo_ref_zip_grad)
)
print(
    "Jednoznačne ZIP + država → grad veze:",
    len(geo_ref_zip_drzava)
)

Jednoznačne ZIP + grad → država veze: 19515
Jednoznačne ZIP + država → grad veze: 18538


### Zaključak čišćenja tablice `geolocation`

Sve ispravke provedene su tek nakon validacije nad podacima iz tablice `geolocation`. Izvorni atributi ostali su sačuvani, a daljnja obrada koristi očišćene vrijednosti. Višeznačne veze koje mogu predstavljati različita naselja ili administrativne cjeline nisu automatski objedinjene.

# 2. Čišćenje tablice `customers`

In [533]:
print("Broj redaka:", len(customers_clean))
print(
    "Broj potpuno dupliciranih redaka:",
    customers_clean.duplicated().sum()
)
print(
    "Broj dupliciranih customer_id:",
    customers_clean["customer_id"].duplicated().sum()
)
print(
    "Broj jedinstvenih customer_unique_id:",
    customers_clean["customer_unique_id"].nunique()
)
print("\nNedostajuće vrijednosti:")
print(customers_clean.isna().sum())

Broj redaka: 99441
Broj potpuno dupliciranih redaka: 0
Broj dupliciranih customer_id: 0
Broj jedinstvenih customer_unique_id: 96096

Nedostajuće vrijednosti:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


## Standardizacija lokacijskih atributa kupaca

In [534]:
customers_clean["customer_city_std"] = (
    customers_clean["customer_city"]
    .apply(standardiziraj_tekst)
)

customers_clean["customer_state_std"] = (
    customers_clean["customer_state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

customers_clean["customer_city_clean"] = (
    customers_clean["customer_city_std"].copy()
)

customers_clean["customer_state_clean"] = (
    customers_clean["customer_state_std"].copy()
)

customers_clean["customer_city_corrected"] = False
customers_clean["customer_state_corrected"] = False

customers_clean["customer_correction_method"] = pd.Series(
    "nije mijenjano",
    index=customers_clean.index,
    dtype="string"
)

## Validirana standardizacija inačica São Paula kod kupaca

Promjena se dopušta samo uz državu `SP` i ZIP prefiks koji u očišćenoj tablici `geolocation` već ima potvrđeni grad `sao paulo`.

In [535]:
customers_clean["customer_city_compact"] = (
    customers_clean["customer_city_clean"]
    .apply(kompaktni_naziv)
)

customers_clean["udaljenost_od_sao_paulo"] = (
    customers_clean["customer_city_compact"]
    .apply(
        lambda naziv: (
            levenshtein_udaljenost(naziv, "saopaulo")
            if pd.notna(naziv)
            else pd.NA
        )
    )
)

maska_sao_paulo_customers = (
    (
        customers_clean["customer_state_clean"] == "SP"
    )
    &
    (
        customers_clean["customer_zip_code_prefix"].isin(
            zipovi_s_potvrdenim_sao_paulo
        )
    )
    &
    (
        (
            customers_clean["customer_city_clean"].isin(
                poznate_varijante_sao_paulo
            )
        )
        |
        (
            customers_clean["customer_city_compact"] == "saopaulo"
        )
        |
        (
            customers_clean["udaljenost_od_sao_paulo"].le(2)
            &
            customers_clean[
                "customer_city_clean"
            ].str.split().str.len().le(3)
        )
    )
    &
    (
        customers_clean["customer_city_clean"] != "sao paulo"
    )
)

print(
    "Broj validiranih inačica São Paula kod kupaca:",
    int(maska_sao_paulo_customers.sum())
)

Broj validiranih inačica São Paula kod kupaca: 0


In [536]:
customers_clean.loc[
    maska_sao_paulo_customers,
    "customer_city_clean"
] = "sao paulo"

customers_clean.loc[
    maska_sao_paulo_customers,
    "customer_city_corrected"
] = True

customers_clean.loc[
    maska_sao_paulo_customers,
    "customer_correction_method"
] = (
    "inačica São Paula potvrđena geolocation podacima"
)

## Validacija savezne države prema jednoznačnoj vezi ZIP + grad

In [537]:
customers_clean = customers_clean.merge(
    geo_ref_zip_grad,
    left_on=[
        "customer_zip_code_prefix",
        "customer_city_clean"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_city_clean"
    ],
    how="left",
    validate="many_to_one"
)

customers_clean = customers_clean.drop(
    columns=[
        "geolocation_zip_code_prefix",
        "geolocation_city_clean"
    ]
)

maska_ispravka_customer_state = (
    customers_clean["referentna_drzava"].notna()
    &
    (
        customers_clean["customer_state_clean"]
        != customers_clean["referentna_drzava"]
    )
)

print(
    "Broj validiranih ispravaka države kod kupaca:",
    int(maska_ispravka_customer_state.sum())
)

Broj validiranih ispravaka države kod kupaca: 0


In [538]:
customers_clean.loc[
    maska_ispravka_customer_state,
    "customer_state_clean"
] = customers_clean.loc[
    maska_ispravka_customer_state,
    "referentna_drzava"
]

customers_clean.loc[
    maska_ispravka_customer_state,
    "customer_state_corrected"
] = True

customers_clean.loc[
    maska_ispravka_customer_state,
    "customer_correction_method"
] = (
    "država potvrđena jednoznačnom vezom ZIP + grad"
)

## Završna provjera povezivosti kupaca

In [539]:
geo_ref_puna_lokacija = (
    geolocation_clean[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city_clean",
            "geolocation_state_clean"
        ]
    ]
    .drop_duplicates()
)

customers_povezivost = customers_clean.merge(
    geo_ref_puna_lokacija,
    left_on=[
        "customer_zip_code_prefix",
        "customer_city_clean",
        "customer_state_clean"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_city_clean",
        "geolocation_state_clean"
    ],
    how="left",
    indicator=True
)

customers_clean["customer_geolocation_matched"] = (
    customers_povezivost["_merge"]
    .eq("both")
    .to_numpy()
)

nepovezani_customers = (
    customers_clean[
        ~customers_clean["customer_geolocation_matched"]
    ]
    .copy()
)

zipovi_geolocation = set(
    geolocation_clean["geolocation_zip_code_prefix"]
)

nepovezani_customers["zip_postoji"] = (
    nepovezani_customers["customer_zip_code_prefix"]
    .isin(zipovi_geolocation)
)

print(customers_povezivost["_merge"].value_counts())
print(
    "\nPostotak povezanih kupaca:",
    round(
        customers_clean[
            "customer_geolocation_matched"
        ].mean() * 100,
        2
    ),
    "%"
)
print(
    "Nepovezani kupci čiji ZIP postoji:",
    int(nepovezani_customers["zip_postoji"].sum())
)
print(
    "Nepovezani kupci čiji ZIP ne postoji:",
    int((~nepovezani_customers["zip_postoji"]).sum())
)

_merge
both          99138
left_only       303
right_only        0
Name: count, dtype: int64

Postotak povezanih kupaca: 99.7 %
Nepovezani kupci čiji ZIP postoji: 25
Nepovezani kupci čiji ZIP ne postoji: 278


### Zaključak čišćenja tablice `customers`

Kupci nisu uklanjani zbog nepovezivosti. Promjene grada i države dopuštene su samo kada ih je bilo moguće potvrditi očišćenim zapisima iz `geolocation`. Nepovezani zapisi ostali su sačuvani i označeni indikatorom povezivosti.

# 3. Čišćenje tablice `sellers`

In [540]:
print("Broj redaka:", len(sellers_clean))
print(
    "Broj potpuno dupliciranih redaka:",
    sellers_clean.duplicated().sum()
)
print(
    "Broj dupliciranih seller_id:",
    sellers_clean["seller_id"].duplicated().sum()
)
print("\nNedostajuće vrijednosti:")
print(sellers_clean.isna().sum())

Broj redaka: 3095
Broj potpuno dupliciranih redaka: 0
Broj dupliciranih seller_id: 0

Nedostajuće vrijednosti:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


## Standardizacija lokacijskih atributa prodavača

In [541]:
sellers_clean["seller_city_std"] = (
    sellers_clean["seller_city"]
    .apply(standardiziraj_tekst)
)

sellers_clean["seller_state_std"] = (
    sellers_clean["seller_state"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sellers_clean["seller_city_clean"] = (
    sellers_clean["seller_city_std"].copy()
)

sellers_clean["seller_state_clean"] = (
    sellers_clean["seller_state_std"].copy()
)

sellers_clean["seller_city_corrected"] = False
sellers_clean["seller_state_corrected"] = False

sellers_clean["seller_correction_method"] = pd.Series(
    "nije mijenjano",
    index=sellers_clean.index,
    dtype="string"
)

## Validacija savezne države prema jednoznačnoj vezi ZIP + grad

In [542]:
sellers_clean = sellers_clean.merge(
    geo_ref_zip_grad,
    left_on=[
        "seller_zip_code_prefix",
        "seller_city_clean"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_city_clean"
    ],
    how="left",
    validate="many_to_one"
)

sellers_clean = sellers_clean.drop(
    columns=[
        "geolocation_zip_code_prefix",
        "geolocation_city_clean"
    ]
)

maska_ispravka_seller_state = (
    sellers_clean["referentna_drzava"].notna()
    &
    (
        sellers_clean["seller_state_clean"]
        != sellers_clean["referentna_drzava"]
    )
)

print(
    "Broj validiranih ispravaka države kod prodavača:",
    int(maska_ispravka_seller_state.sum())
)

sellers_clean.loc[
    maska_ispravka_seller_state,
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_city_clean",
        "seller_state",
        "seller_state_clean",
        "referentna_drzava"
    ]
].drop_duplicates()

Broj validiranih ispravaka države kod prodavača: 32


,seller_zip_code_prefix,seller_city,seller_city_clean,seller_state,seller_state_clean,referentna_drzava
70,95076,caxias do sul,caxias do sul,SP,SP,RS
114,21210,rio de janeiro,rio de janeiro,RN,RN,RJ
198,85301,laranjeiras do sul,laranjeiras do sul,SP,SP,PR
206,87360,goioere,goioere,SP,SP,PR
275,86170,sertanopolis,sertanopolis,SP,SP,PR
311,85960,marechal candido rondon,marechal candido rondon,SP,SP,PR
364,22783,rio de janeiro,rio de janeiro,SP,SP,RJ
424,36512,tocantins,tocantins,SP,SP,MG
513,27277,volta redonda,volta redonda,SP,SP,RJ
549,31570,belo horizonte,belo horizonte,SP,SP,MG


In [543]:
sellers_clean.loc[
    maska_ispravka_seller_state,
    "seller_state_clean"
] = sellers_clean.loc[
    maska_ispravka_seller_state,
    "referentna_drzava"
]

sellers_clean.loc[
    maska_ispravka_seller_state,
    "seller_state_corrected"
] = True

sellers_clean.loc[
    maska_ispravka_seller_state,
    "seller_correction_method"
] = (
    "država potvrđena jednoznačnom vezom ZIP + grad"
)

## Validacija formatnog čišćenja naziva grada

Formatna promjena prvo se zapisuje kao kandidat. Primjenjuje se samo ako dobiveni grad postoji u `geolocation` uz isti ZIP prefiks i saveznu državu.

In [544]:
def ukloni_dodatke_iz_grada(grad, drzava):
    if pd.isna(grad):
        return pd.NA

    rezultat = str(grad).strip()

    if pd.notna(drzava):
        rezultat = re.sub(
            rf"\s+{re.escape(str(drzava).lower())}$",
            "",
            rezultat
        ).strip()

    rezultat = re.sub(
        r"\s+brasil$",
        "",
        rezultat
    ).strip()

    administrativni_dodaci = [
        "rio grande do sul",
        "minas gerais",
        "sao paulo",
        "rio de janeiro",
        "santa catarina"
    ]

    for dodatak in administrativni_dodaci:
        if rezultat != dodatak:
            rezultat = re.sub(
                rf"\s+{re.escape(dodatak)}$",
                "",
                rezultat
            ).strip()

    return rezultat


sellers_clean["seller_city_format_candidate"] = (
    sellers_clean.apply(
        lambda redak: ukloni_dodatke_iz_grada(
            redak["seller_city_clean"],
            redak["seller_state_clean"]
        ),
        axis=1
    )
)

maska_formatni_kandidati = (
    sellers_clean["seller_city_format_candidate"]
    != sellers_clean["seller_city_clean"]
)

geo_pune_lokacije = set(
    geolocation_clean[
        [
            "geolocation_zip_code_prefix",
            "geolocation_state_clean",
            "geolocation_city_clean"
        ]
    ]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)

maska_formatna_ispravka_potvrdena = (
    sellers_clean.apply(
        lambda redak: (
            redak["seller_zip_code_prefix"],
            redak["seller_state_clean"],
            redak["seller_city_format_candidate"]
        ) in geo_pune_lokacije,
        axis=1
    )
    &
    maska_formatni_kandidati
)

print(
    "Broj kandidata za formatnu promjenu:",
    int(maska_formatni_kandidati.sum())
)
print(
    "Broj formatnih promjena potvrđenih geolocation podacima:",
    int(maska_formatna_ispravka_potvrdena.sum())
)

sellers_clean.loc[
    maska_formatni_kandidati,
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_city_clean",
        "seller_city_format_candidate",
        "seller_state_clean"
    ]
].assign(
    potvrdeno_geolocation=(
        maska_formatna_ispravka_potvrdena[
            maska_formatni_kandidati
        ].to_numpy()
    )
)

Broj kandidata za formatnu promjenu: 26
Broj formatnih promjena potvrđenih geolocation podacima: 23


,seller_zip_code_prefix,seller_city,seller_city_clean,seller_city_format_candidate,seller_state_clean,potvrdeno_geolocation
78,88501,lages - sc,lages sc,lages,SC,True
237,15350,auriflama/sp,auriflama sp,auriflama,SP,True
246,3407,sao paulo / sao paulo,sao paulo sao paulo,sao paulo,SP,True
551,93310,"novo hamburgo, rio grande do sul, brasil",novo hamburgo rio grande do sul brasil,novo hamburgo,RS,True
622,29142,cariacica / es,cariacica es,cariacica,ES,True
795,71906,brasilia df,brasilia df,brasilia,DF,True
826,5353,sao paulo - sp,sao paulo sp,sao paulo,SP,True
869,9726,sbc/sp,sbc sp,sbc,SP,False
945,9230,santo andre/sao paulo,santo andre sao paulo,santo andre,SP,True
1004,3363,sp / sp,sp sp,sp,SP,False


In [545]:
sellers_clean.loc[
    maska_formatna_ispravka_potvrdena,
    "seller_city_clean"
] = sellers_clean.loc[
    maska_formatna_ispravka_potvrdena,
    "seller_city_format_candidate"
]

sellers_clean.loc[
    maska_formatna_ispravka_potvrdena,
    "seller_city_corrected"
] = True

sellers_clean.loc[
    maska_formatna_ispravka_potvrdena,
    "seller_correction_method"
] = (
    "formatna ispravka potvrđena geolocation podacima"
)

## Validacija predloženih tipografskih i semantičkih ispravaka

Sva ranije definirana pravila objedinjena su u jednu tablicu prijedloga. Nijedno se pravilo ne primjenjuje prije provjere `geolocation` podataka.

In [546]:
predlozene_ispravke_sellers = pd.DataFrame(
    [
        (63020, "CE", "juzeiro do norte", "juazeiro do norte", "tipografska pogreška"),
        (31255, "MG", "belo horizont", "belo horizonte", "tipografska pogreška"),
        (83040, "PR", "sao jose dos pinhas", "sao jose dos pinhais", "tipografska pogreška"),
        (85802, "PR", "cascavael", "cascavel", "tipografska pogreška"),
        (87140, "PR", "paincandu", "paicandu", "tipografska pogreška"),
        (88056, "SC", "floranopolis", "florianopolis", "tipografska pogreška"),
        (88330, "SC", "balenario camboriu", "balneario camboriu", "tipografska pogreška"),
        (2051, "SP", "sao pauo", "sao paulo", "tipografska pogreška"),
        (3581, "SP", "sao paulop", "sao paulo", "tipografska pogreška"),
        (8050, "SP", "sao paluo", "sao paulo", "tipografska pogreška"),
        (3363, "SP", "sp", "sao paulo", "kratica potvrđena ZIP-om"),
        (4776, "SP", "sp", "sao paulo", "kratica potvrđena ZIP-om"),
        (5141, "SP", "sp", "sao paulo", "kratica potvrđena ZIP-om"),
        (12903, "SP", "sp", "sao paulo", "kratica potvrđena ZIP-om"),
        (16021, "SP", "sp", "sao paulo", "kratica potvrđena ZIP-om"),
        (7077, "SP", "garulhos", "guarulhos", "tipografska pogreška"),
        (8710, "SP", "mogi das cruses", "mogi das cruzes", "tipografska pogreška"),
        (9190, "SP", "sando andre", "santo andre", "tipografska pogreška"),
        (9687, "SP", "ao bernardo do campo", "sao bernardo do campo", "tipografska pogreška"),
        (9721, "SP", "sao bernardo do capo", "sao bernardo do campo", "tipografska pogreška"),
        (9726, "SP", "sbc", "sao bernardo do campo", "kratica potvrđena ZIP-om"),
        (9861, "SP", "sbc", "sao bernardo do campo", "kratica potvrđena ZIP-om"),
        (13660, "SP", "portoferreira", "porto ferreira", "nedostaje razmak"),
        (13720, "SP", "scao jose do rio pardo", "sao jose do rio pardo", "tipografska pogreška"),
        (14027, "SP", "ribeirao pretp", "ribeirao preto", "tipografska pogreška"),
        (14078, "SP", "robeirao preto", "ribeirao preto", "tipografska pogreška"),
        (14085, "SP", "riberao preto", "ribeirao preto", "tipografska pogreška"),
        (15014, "SP", "s jose do rio preto", "sao jose do rio preto", "skraćen zapis"),
        (15051, "SP", "sao jose do rio pret", "sao jose do rio preto", "tipografska pogreška"),
        (45816, "BA", "arraial d ajuda porto seguro", "arraial d ajuda", "uklanjanje administrativnog dodatka"),
        (6764, "SP", "tabao da serra", "taboao da serra", "tipografska pogreška"),
        (89900, "SC", "sao miguel d oeste", "sao miguel do oeste", "tipografska varijanta"),
        (48602, "BA", "bahia", "paulo afonso", "u grad unesena država"),
        (35660, "MG", "centro", "para de minas", "generički lokalitet"),
        (37165, "MG", "minas gerais", "campo do meio", "u grad unesena država"),
        (87025, "PR", "vendas creditparts com br", "maringa", "u grad unesena e-mail adresa"),
        (87083, "PR", "parana", "maringa", "u grad unesena država"),
        (22790, "RJ", "04482255", "rio de janeiro", "u grad unesena brojčana vrijednost"),
        (88135, "SC", "santa catarina", "palhoca", "u grad unesena država")
    ],
    columns=[
        "zip",
        "drzava",
        "pogresan_grad",
        "predlozeni_grad",
        "vrsta_ispravke"
    ]
)

predlozene_ispravke_sellers

,zip,drzava,pogresan_grad,predlozeni_grad,vrsta_ispravke
0,63020,CE,juzeiro do norte,juazeiro do norte,tipografska pogreška
1,31255,MG,belo horizont,belo horizonte,tipografska pogreška
2,83040,PR,sao jose dos pinhas,sao jose dos pinhais,tipografska pogreška
3,85802,PR,cascavael,cascavel,tipografska pogreška
4,87140,PR,paincandu,paicandu,tipografska pogreška
5,88056,SC,floranopolis,florianopolis,tipografska pogreška
6,88330,SC,balenario camboriu,balneario camboriu,tipografska pogreška
7,2051,SP,sao pauo,sao paulo,tipografska pogreška
8,3581,SP,sao paulop,sao paulo,tipografska pogreška
9,8050,SP,sao paluo,sao paulo,tipografska pogreška


In [547]:
ucestalost_geo_gradova = (
    geolocation_clean
    .groupby(
        [
            "geolocation_zip_code_prefix",
            "geolocation_state_clean",
            "geolocation_city_clean"
        ]
    )
    .size()
    .reset_index(name="broj_pojavljivanja")
)


def validiraj_ispravku_seller(redak):
    kandidati = (
        ucestalost_geo_gradova[
            (
                ucestalost_geo_gradova[
                    "geolocation_zip_code_prefix"
                ] == redak["zip"]
            )
            &
            (
                ucestalost_geo_gradova[
                    "geolocation_state_clean"
                ] == redak["drzava"]
            )
        ]
        .sort_values(
            "broj_pojavljivanja",
            ascending=False
        )
        .copy()
    )

    if kandidati.empty:
        return pd.Series(
            {
                "gradovi_u_geolocation": [],
                "najcesci_grad": pd.NA,
                "predlozeni_grad_postoji": False,
                "broj_predlozenog_grada": 0,
                "odluka": "ne primijeniti",
                "obrazlozenje_validacije": (
                    "Za isti ZIP i državu nema zapisa u geolocation."
                )
            }
        )

    gradovi = list(
        zip(
            kandidati["geolocation_city_clean"],
            kandidati["broj_pojavljivanja"]
        )
    )

    najcesci_grad = kandidati.iloc[0][
        "geolocation_city_clean"
    ]

    maska_predlozeni = (
        kandidati["geolocation_city_clean"]
        == redak["predlozeni_grad"]
    )

    predlozeni_postoji = bool(maska_predlozeni.any())

    broj_predlozenog = int(
        kandidati.loc[
            maska_predlozeni,
            "broj_pojavljivanja"
        ].sum()
    ) if predlozeni_postoji else 0

    if not predlozeni_postoji:
        odluka = "ne primijeniti"
        obrazlozenje = (
            "Predloženi grad ne postoji uz isti ZIP i državu."
        )
    elif len(kandidati) == 1:
        odluka = "primijeniti"
        obrazlozenje = (
            "ZIP i država jednoznačno upućuju na predloženi grad."
        )
    elif najcesci_grad == redak["predlozeni_grad"]:
        odluka = "primijeniti"
        obrazlozenje = (
            "Predloženi grad postoji i najčešći je za isti ZIP i državu."
        )
    else:
        odluka = "dodatno provjeriti"
        obrazlozenje = (
            "Predloženi grad postoji, ali nije jedini ni najčešći."
        )

    return pd.Series(
        {
            "gradovi_u_geolocation": gradovi,
            "najcesci_grad": najcesci_grad,
            "predlozeni_grad_postoji": predlozeni_postoji,
            "broj_predlozenog_grada": broj_predlozenog,
            "odluka": odluka,
            "obrazlozenje_validacije": obrazlozenje
        }
    )


rezultati_validacije_sellers = (
    predlozene_ispravke_sellers
    .apply(
        validiraj_ispravku_seller,
        axis=1
    )
)

validacija_ispravaka_sellers = pd.concat(
    [
        predlozene_ispravke_sellers.reset_index(drop=True),
        rezultati_validacije_sellers.reset_index(drop=True)
    ],
    axis=1
)

validacija_ispravaka_sellers

,zip,drzava,pogresan_grad,predlozeni_grad,vrsta_ispravke,gradovi_u_geolocation,najcesci_grad,predlozeni_grad_postoji,broj_predlozenog_grada,odluka,obrazlozenje_validacije
0,63020,CE,juzeiro do norte,juazeiro do norte,tipografska pogreška,"[(juazeiro do norte, 41)]",juazeiro do norte,True,41,primijeniti,ZIP i država jednoznačno upućuju na predloženi...
1,31255,MG,belo horizont,belo horizonte,tipografska pogreška,"[(belo horizonte, 162)]",belo horizonte,True,162,primijeniti,ZIP i država jednoznačno upućuju na predloženi...
2,83040,PR,sao jose dos pinhas,sao jose dos pinhais,tipografska pogreška,"[(sao jose dos pinhais, 70)]",sao jose dos pinhais,True,70,primijeniti,ZIP i država jednoznačno upućuju na predloženi...
3,85802,PR,cascavael,cascavel,tipografska pogreška,"[(cascavel, 106)]",cascavel,True,106,primijeniti,ZIP i država jednoznačno upućuju na predloženi...
4,87140,PR,paincandu,paicandu,tipografska pogreška,"[(paicandu, 100)]",paicandu,True,100,primijeniti,ZIP i država jednoznačno upućuju na predloženi...
5,88056,SC,floranopolis,florianopolis,tipografska pogreška,"[(florianopolis, 101)]",florianopolis,True,101,primijeniti,ZIP i država jednoznačno upućuju na predloženi...
6,88330,SC,balenario camboriu,balneario camboriu,tipografska pogreška,"[(balneario camboriu, 365), (camboriu, 1)]",balneario camboriu,True,365,primijeniti,Predloženi grad postoji i najčešći je za isti ...
7,2051,SP,sao pauo,sao paulo,tipografska pogreška,"[(sao paulo, 14)]",sao paulo,True,14,primijeniti,ZIP i država jednoznačno upućuju na predloženi...
8,3581,SP,sao paulop,sao paulo,tipografska pogreška,"[(sao paulo, 23)]",sao paulo,True,23,primijeniti,ZIP i država jednoznačno upućuju na predloženi...
9,8050,SP,sao paluo,sao paulo,tipografska pogreška,"[(sao paulo, 160)]",sao paulo,True,160,primijeniti,ZIP i država jednoznačno upućuju na predloženi...


In [548]:
print("Raspodjela odluka:")
print(
    validacija_ispravaka_sellers[
        "odluka"
    ].value_counts(dropna=False)
)

potvrdene_ispravke_sellers = (
    validacija_ispravaka_sellers[
        validacija_ispravaka_sellers["odluka"]
        == "primijeniti"
    ]
    .copy()
)

print(
    "\nBroj pravila potvrđenih geolocation podacima:",
    len(potvrdene_ispravke_sellers)
)

potvrdene_ispravke_sellers[
    [
        "zip",
        "drzava",
        "pogresan_grad",
        "predlozeni_grad",
        "gradovi_u_geolocation",
        "obrazlozenje_validacije"
    ]
]

Raspodjela odluka:
odluka
primijeniti       37
ne primijeniti     2
Name: count, dtype: int64

Broj pravila potvrđenih geolocation podacima: 37


,zip,drzava,pogresan_grad,predlozeni_grad,gradovi_u_geolocation,obrazlozenje_validacije
0,63020,CE,juzeiro do norte,juazeiro do norte,"[(juazeiro do norte, 41)]",ZIP i država jednoznačno upućuju na predloženi...
1,31255,MG,belo horizont,belo horizonte,"[(belo horizonte, 162)]",ZIP i država jednoznačno upućuju na predloženi...
2,83040,PR,sao jose dos pinhas,sao jose dos pinhais,"[(sao jose dos pinhais, 70)]",ZIP i država jednoznačno upućuju na predloženi...
3,85802,PR,cascavael,cascavel,"[(cascavel, 106)]",ZIP i država jednoznačno upućuju na predloženi...
4,87140,PR,paincandu,paicandu,"[(paicandu, 100)]",ZIP i država jednoznačno upućuju na predloženi...
5,88056,SC,floranopolis,florianopolis,"[(florianopolis, 101)]",ZIP i država jednoznačno upućuju na predloženi...
6,88330,SC,balenario camboriu,balneario camboriu,"[(balneario camboriu, 365), (camboriu, 1)]",Predloženi grad postoji i najčešći je za isti ...
7,2051,SP,sao pauo,sao paulo,"[(sao paulo, 14)]",ZIP i država jednoznačno upućuju na predloženi...
8,3581,SP,sao paulop,sao paulo,"[(sao paulo, 23)]",ZIP i država jednoznačno upućuju na predloženi...
9,8050,SP,sao paluo,sao paulo,"[(sao paulo, 160)]",ZIP i država jednoznačno upućuju na predloženi...


In [549]:
selleri_za_validiranu_ispravku = (
    sellers_clean
    .merge(
        potvrdene_ispravke_sellers[
            [
                "zip",
                "drzava",
                "pogresan_grad",
                "predlozeni_grad",
                "obrazlozenje_validacije"
            ]
        ],
        left_on=[
            "seller_zip_code_prefix",
            "seller_state_clean",
            "seller_city_clean"
        ],
        right_on=[
            "zip",
            "drzava",
            "pogresan_grad"
        ],
        how="inner"
    )
)

print(
    "Broj prodavača koji će biti izmijenjeni:",
    len(selleri_za_validiranu_ispravku)
)

selleri_za_validiranu_ispravku[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_city_clean",
        "seller_state_clean",
        "predlozeni_grad",
        "obrazlozenje_validacije"
    ]
]

Broj prodavača koji će biti izmijenjeni: 35


,seller_id,seller_zip_code_prefix,seller_city,seller_city_clean,seller_state_clean,predlozeni_grad,obrazlozenje_validacije
0,5e49e3a44bdeb5aab2684258bbd4f525,88330,balenario camboriu,balenario camboriu,SC,balneario camboriu,Predloženi grad postoji i najčešći je za isti ...
1,5444b12c82f21c923f2639ebc722c1ea,2051,sao pauo,sao pauo,SP,sao paulo,ZIP i država jednoznačno upućuju na predloženi...
2,4221a7df464f1fe2955934e30ff3a5a1,48602,bahia,bahia,BA,paulo afonso,ZIP i država jednoznačno upućuju na predloženi...
3,34675b28b826facfde119785bf287dc1,85802,cascavael,cascavael,PR,cascavel,ZIP i država jednoznačno upućuju na predloženi...
4,ceb7b4fb9401cd378de7886317ad1b47,22790,04482255,04482255,RJ,rio de janeiro,ZIP i država jednoznačno upućuju na predloženi...
5,bb7ad8a45c027be8ab075b8e465f8ca0,88056,floranopolis,floranopolis,SC,florianopolis,ZIP i država jednoznačno upućuju na predloženi...
6,11938fcc75f6902fea1d0a6f72b54a57,89900,sao miguel d'oeste,sao miguel d oeste,SC,sao miguel do oeste,ZIP i država jednoznačno upućuju na predloženi...
7,596849622429351f47b32e6cae1055ff,8710,mogi das cruses,mogi das cruses,SP,mogi das cruzes,ZIP i država jednoznačno upućuju na predloženi...
8,4aba391bc3b88717ce08eb11e44937b2,45816,arraial d'ajuda (porto seguro),arraial d ajuda porto seguro,BA,arraial d ajuda,Predloženi grad postoji i najčešći je za isti ...
9,701938c450705b8ae65fc923b70f35c7,15014,s jose do rio preto,s jose do rio preto,SP,sao jose do rio preto,ZIP i država jednoznačno upućuju na predloženi...


In [550]:
for _, ispravka in potvrdene_ispravke_sellers.iterrows():
    maska = (
        (
            sellers_clean["seller_zip_code_prefix"]
            == ispravka["zip"]
        )
        &
        (
            sellers_clean["seller_state_clean"]
            == ispravka["drzava"]
        )
        &
        (
            sellers_clean["seller_city_clean"]
            == ispravka["pogresan_grad"]
        )
    )

    indeksi = sellers_clean.index[maska]

    sellers_clean.loc[
        maska,
        "seller_city_clean"
    ] = ispravka["predlozeni_grad"]

    sellers_clean.loc[
        maska,
        "seller_city_corrected"
    ] = True

    for indeks in indeksi:
        prethodna_metoda = sellers_clean.at[
            indeks,
            "seller_correction_method"
        ]

        dodatna_metoda = (
            "ispravka grada potvrđena geolocation podacima"
        )

        if (
            pd.isna(prethodna_metoda)
            or prethodna_metoda == "nije mijenjano"
        ):
            nova_metoda = dodatna_metoda
        elif dodatna_metoda not in prethodna_metoda:
            nova_metoda = (
                prethodna_metoda
                + "; "
                + dodatna_metoda
            )
        else:
            nova_metoda = prethodna_metoda

        sellers_clean.at[
            indeks,
            "seller_correction_method"
        ] = nova_metoda

## Završna provjera prodavača

In [551]:
seller_geo_nakon = sellers_clean.merge(
    geo_ref_puna_lokacija,
    left_on=[
        "seller_zip_code_prefix",
        "seller_city_clean",
        "seller_state_clean"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_city_clean",
        "geolocation_state_clean"
    ],
    how="left",
    indicator=True
)

sellers_clean["seller_geolocation_matched"] = (
    seller_geo_nakon["_merge"]
    .eq("both")
    .to_numpy()
)

nepovezani_sellers_nakon = (
    sellers_clean[
        ~sellers_clean["seller_geolocation_matched"]
    ]
    .copy()
)

nepovezani_sellers_nakon["zip_postoji"] = (
    nepovezani_sellers_nakon[
        "seller_zip_code_prefix"
    ].isin(zipovi_geolocation)
)

sellers_clean["seller_city_invalid_format"] = (
    sellers_clean["seller_city_clean"].str.fullmatch(
        r"\d+",
        na=False
    )
    |
    sellers_clean["seller_city"].str.contains(
        "@",
        na=False
    )
)

print(seller_geo_nakon["_merge"].value_counts())
print(
    "\nPostotak povezanih prodavača:",
    round(
        sellers_clean[
            "seller_geolocation_matched"
        ].mean() * 100,
        2
    ),
    "%"
)
print(
    "Broj nepovezanih prodavača:",
    len(nepovezani_sellers_nakon)
)
print(
    "Nepovezani prodavači čiji ZIP postoji:",
    int(nepovezani_sellers_nakon["zip_postoji"].sum())
)
print(
    "Nepovezani prodavači čiji ZIP ne postoji:",
    int((~nepovezani_sellers_nakon["zip_postoji"]).sum())
)

_merge
both          3057
left_only       38
right_only       0
Name: count, dtype: int64

Postotak povezanih prodavača: 98.77 %
Broj nepovezanih prodavača: 38
Nepovezani prodavači čiji ZIP postoji: 31
Nepovezani prodavači čiji ZIP ne postoji: 7


In [552]:
pregled_promjena_sellers = (
    sellers_clean[
        sellers_clean["seller_city_corrected"]
        |
        sellers_clean["seller_state_corrected"]
    ][
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_city_std",
            "seller_city_clean",
            "seller_state",
            "seller_state_std",
            "seller_state_clean",
            "seller_city_corrected",
            "seller_state_corrected",
            "seller_correction_method"
        ]
    ]
    .sort_values(
        [
            "seller_state_clean",
            "seller_zip_code_prefix"
        ]
    )
)

pregled_promjena_sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_city_std,seller_city_clean,seller_state,seller_state_std,seller_state_clean,seller_city_corrected,seller_state_corrected,seller_correction_method
2048,b43d8b707f887a30f4675391098354a0,44600,ipira,ipira,ipira,SP,SP,BA,False,True,država potvrđena jednoznačnom vezom ZIP + grad
874,4aba391bc3b88717ce08eb11e44937b2,45816,arraial d'ajuda (porto seguro),arraial d ajuda porto seguro,arraial d ajuda,BA,BA,BA,True,False,ispravka grada potvrđena geolocation podacima
405,4221a7df464f1fe2955934e30ff3a5a1,48602,bahia,bahia,paulo afonso,BA,BA,BA,True,False,ispravka grada potvrđena geolocation podacima
1013,d4828feed4cda2cd9d6b37899a95a71e,63020,juzeiro do norte,juzeiro do norte,juazeiro do norte,CE,CE,CE,True,False,ispravka grada potvrđena geolocation podacima
795,4170f6a9367e6b5d6401d03ce6d54569,71906,brasilia df,brasilia df,brasilia,DF,DF,DF,True,False,formatna ispravka potvrđena geolocation podacima
...,...,...,...,...,...,...,...,...,...,...,...
1920,01fd077212124329bac32490e8ef80d9,14079,ribeirao preto / sao paulo,ribeirao preto sao paulo,ribeirao preto,SP,SP,SP,True,False,formatna ispravka potvrđena geolocation podacima
3026,17e34d8224d27a541263c4c64b11a56b,14085,riberao preto,riberao preto,ribeirao preto,SP,SP,SP,True,False,ispravka grada potvrđena geolocation podacima
979,701938c450705b8ae65fc923b70f35c7,15014,s jose do rio preto,s jose do rio preto,sao jose do rio preto,SP,SP,SP,True,False,ispravka grada potvrđena geolocation podacima
1937,28fb0829fd777877cdc350d25cb5bd5b,15051,sao jose do rio pret,sao jose do rio pret,sao jose do rio preto,SP,SP,SP,True,False,ispravka grada potvrđena geolocation podacima


### Zaključak čišćenja tablice `sellers`

Država, formatne izmjene i tipografske ispravke primijenjene su samo nakon provjere pomoću očišćenih `geolocation` podataka. Pravila koja nisu mogla biti potvrđena nisu primijenjena. Nepovezani prodavači ostali su sačuvani i označeni indikatorom povezivosti.

# 4. Čišćenje tablice `products`

Tijekom čišćenja tablice `products`:

- provjeravaju se duplikati i nedostajuće vrijednosti;
- nedostajuće kategorije označavaju se vrijednošću `unknown`;
- dopunjava se referentna tablica prijevoda kategorija;
- nevaljane nulte težine pretvaraju se u nedostajuće vrijednosti u novom očišćenom atributu;
- izračunava se volumen proizvoda;
- statistički outlieri samo se označavaju, bez uklanjanja zapisa.

In [553]:
print("Broj redaka:", len(products_clean))
print(
    "Broj potpuno dupliciranih redaka:",
    products_clean.duplicated().sum()
)
print(
    "Broj dupliciranih product_id:",
    products_clean["product_id"].duplicated().sum()
)
print("\nNedostajuće vrijednosti:")
print(products_clean.isna().sum())

Broj redaka: 32951
Broj potpuno dupliciranih redaka: 0
Broj dupliciranih product_id: 0

Nedostajuće vrijednosti:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


## Standardizacija kategorija proizvoda

Proizvodi bez izvorno definirane kategorije zadržavaju se u skupu podataka, a u očišćenom atributu dobivaju vrijednost `unknown`.

In [554]:
products_clean["product_category_name_clean"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
    .astype("string")
    .str.strip()
    .str.lower()
)

products_clean["missing_category"] = (
    products_clean["product_category_name"].isna()
)

## Dopunjavanje referentne tablice prijevoda

U očišćenu referentnu tablicu dodaju se prijevodi za dvije poznate kategorije koje nedostaju te zapis za kategoriju `unknown`. Izvorna tablica ostaje nepromijenjena.

In [555]:
dodatni_prijevodi = pd.DataFrame(
    [
        {
            "product_category_name": "pc_gamer",
            "product_category_name_english": "PC gamer"
        },
        {
            "product_category_name":
                "portateis_cozinha_e_preparadores_de_alimentos",
            "product_category_name_english":
                "Portable kitchen appliances"
        },
        {
            "product_category_name": "unknown",
            "product_category_name_english": "Unknown category"
        }
    ]
)

category_translation_clean = (
    pd.concat(
        [
            category_translation_clean,
            dodatni_prijevodi
        ],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["product_category_name"],
        keep="first"
    )
    .reset_index(drop=True)
)

assert (
    category_translation_clean[
        "product_category_name"
    ].duplicated().sum()
    == 0
)

## Povezivanje proizvoda s engleskim nazivima kategorija

In [556]:
products_clean = products_clean.drop(
    columns=[
        stupac
        for stupac in [
            "product_category_name_english",
            "product_category_name_translation",
            "product_category_name_y"
        ]
        if stupac in products_clean.columns
    ],
    errors="ignore"
)

products_clean = products_clean.merge(
    category_translation_clean[
        [
            "product_category_name",
            "product_category_name_english"
        ]
    ],
    left_on="product_category_name_clean",
    right_on="product_category_name",
    how="left",
    validate="many_to_one",
    suffixes=("", "_translation")
)

if "product_category_name_translation" in products_clean.columns:
    products_clean = products_clean.drop(
        columns=["product_category_name_translation"]
    )

assert (
    products_clean["product_category_name_english"]
    .isna()
    .sum()
    == 0
)

## Čišćenje fizičkih karakteristika proizvoda

Izvorna težina ostaje sačuvana. Vrijednost `0 g` smatra se nevaljanom za fizički proizvod i u novom atributu `product_weight_g_clean` zamjenjuje se nedostajućom vrijednošću. 

In [557]:
fizicke_mjere = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_clean["missing_dimensions"] = (
    products_clean[fizicke_mjere]
    .isna()
    .any(axis=1)
)

products_clean["product_weight_invalid"] = (
    products_clean["product_weight_g"].eq(0)
)

products_clean["product_weight_g_clean"] = (
    products_clean["product_weight_g"]
    .mask(products_clean["product_weight_invalid"])
)

products_clean["product_volume_cm3"] = (
    products_clean["product_length_cm"]
    * products_clean["product_height_cm"]
    * products_clean["product_width_cm"]
)

## Označavanje statističkih outliera

IQR metoda koristi se samo za stvaranje indikatora kvalitete. Outlieri se ne uklanjaju jer sami po sebi nisu dokaz pogrešnog podatka.

In [558]:
q1_weight = products_clean["product_weight_g_clean"].quantile(0.25)
q3_weight = products_clean["product_weight_g_clean"].quantile(0.75)
iqr_weight = q3_weight - q1_weight
lower_weight = q1_weight - 1.5 * iqr_weight
upper_weight = q3_weight + 1.5 * iqr_weight

products_clean["weight_low_outlier"] = (
    products_clean["product_weight_g_clean"] < lower_weight
)
products_clean["weight_high_outlier"] = (
    products_clean["product_weight_g_clean"] > upper_weight
)
products_clean["weight_outlier"] = (
    products_clean["weight_low_outlier"]
    | products_clean["weight_high_outlier"]
)

q1_volume = products_clean["product_volume_cm3"].quantile(0.25)
q3_volume = products_clean["product_volume_cm3"].quantile(0.75)
iqr_volume = q3_volume - q1_volume
lower_volume = q1_volume - 1.5 * iqr_volume
upper_volume = q3_volume + 1.5 * iqr_volume

products_clean["volume_low_outlier"] = (
    products_clean["product_volume_cm3"] < lower_volume
)
products_clean["volume_high_outlier"] = (
    products_clean["product_volume_cm3"] > upper_volume
)
products_clean["volume_outlier"] = (
    products_clean["volume_low_outlier"]
    | products_clean["volume_high_outlier"]
)

In [559]:
zavrsna_provjera_products = pd.Series({
    "broj_redaka": len(products_clean),
    "duplicirani_product_id":
        products_clean["product_id"].duplicated().sum(),
    "kategorija_unknown":
        products_clean["missing_category"].sum(),
    "nepotpune_fizicke_mjere":
        products_clean["missing_dimensions"].sum(),
    "nevaljana_nulta_tezina":
        products_clean["product_weight_invalid"].sum(),
    "nedostajuci_engleski_prijevod":
        products_clean[
            "product_category_name_english"
        ].isna().sum(),
    "outlieri_tezine":
        products_clean["weight_outlier"].sum(),
    "outlieri_volumena":
        products_clean["volume_outlier"].sum()
})

zavrsna_provjera_products

broj_redaka                      32951
duplicirani_product_id               0
kategorija_unknown                 610
nepotpune_fizicke_mjere              2
nevaljana_nulta_tezina               4
nedostajuci_engleski_prijevod        0
outlieri_tezine                   4551
outlieri_volumena                 3262
dtype: int64

### Zaključak čišćenja tablice `products`

Nijedan proizvod nije uklonjen. Nedostajuće kategorije označene su vrijednošću `unknown`, referentna tablica prijevoda je dopunjena, nevaljane nulte težine označene su kao nedostajuće u očišćenom atributu, a ekstremne vrijednosti zadržane su uz odgovarajuće indikatore.

# 5. Čišćenje tablice `orders`

Čišćenje obuhvaća pretvorbu vremenskih atributa, označavanje kronoloških nekonzistentnosti, provjeru referencijalnog integriteta i izračun izvedenih vremenskih atributa. Izvorni zapisi ne uklanjaju se zbog uočenih vremenskih odstupanja.

In [560]:
print("Broj redaka:", len(orders_clean))
print(
    "Broj potpuno dupliciranih redaka:",
    orders_clean.duplicated().sum()
)
print(
    "Broj dupliciranih order_id:",
    orders_clean["order_id"].duplicated().sum()
)
print("\nNedostajuće vrijednosti:")
print(orders_clean.isna().sum())

Broj redaka: 99441
Broj potpuno dupliciranih redaka: 0
Broj dupliciranih order_id: 0

Nedostajuće vrijednosti:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


## Pretvorba vremenskih atributa i standardizacija statusa

In [561]:
datum_stupci_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for stupac in datum_stupci_orders:
    orders_clean[stupac] = pd.to_datetime(
        orders_clean[stupac],
        errors="coerce"
    )

orders_clean["order_status"] = (
    orders_clean["order_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

dozvoljeni_statusi = {
    "created",
    "approved",
    "processing",
    "invoiced",
    "shipped",
    "delivered",
    "unavailable",
    "canceled"
}

neocekivani_statusi = (
    set(orders_clean["order_status"].dropna())
    - dozvoljeni_statusi
)

assert len(neocekivani_statusi) == 0

## Označavanje kronoloških nekonzistentnosti

Zapisi se ne ispravljaju automatski. Svaka uočena nekonzistentnost označava se zasebnim indikatorom.

In [562]:
orders_clean["approval_before_purchase"] = (
    orders_clean["order_approved_at"].notna()
    & orders_clean["order_purchase_timestamp"].notna()
    & (
        orders_clean["order_approved_at"]
        < orders_clean["order_purchase_timestamp"]
    )
)

orders_clean["carrier_before_approval"] = (
    orders_clean["order_delivered_carrier_date"].notna()
    & orders_clean["order_approved_at"].notna()
    & (
        orders_clean["order_delivered_carrier_date"]
        < orders_clean["order_approved_at"]
    )
)

orders_clean["customer_before_carrier"] = (
    orders_clean["order_delivered_customer_date"].notna()
    & orders_clean["order_delivered_carrier_date"].notna()
    & (
        orders_clean["order_delivered_customer_date"]
        < orders_clean["order_delivered_carrier_date"]
    )
)

orders_clean["delivery_before_purchase"] = (
    orders_clean["order_delivered_customer_date"].notna()
    & orders_clean["order_purchase_timestamp"].notna()
    & (
        orders_clean["order_delivered_customer_date"]
        < orders_clean["order_purchase_timestamp"]
    )
)

orders_clean["has_timeline_anomaly"] = (
    orders_clean[
        [
            "approval_before_purchase",
            "carrier_before_approval",
            "customer_before_carrier",
            "delivery_before_purchase"
        ]
    ]
    .any(axis=1)
)

## Referencijalni integritet prema kupcima

In [563]:
orders_customers_provjera = orders_clean.merge(
    customers_clean[["customer_id"]],
    on="customer_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

orders_clean["customer_reference_valid"] = (
    orders_customers_provjera["_merge"]
    .eq("both")
    .to_numpy()
)

## Izvedeni vremenski atributi

In [564]:
orders_clean["approval_time_hours"] = (
    (
        orders_clean["order_approved_at"]
        - orders_clean["order_purchase_timestamp"]
    )
    .dt.total_seconds()
    / 3600
)

orders_clean["carrier_wait_days"] = (
    (
        orders_clean["order_delivered_carrier_date"]
        - orders_clean["order_approved_at"]
    )
    .dt.total_seconds()
    / 86400
)

orders_clean["shipping_time_days"] = (
    (
        orders_clean["order_delivered_carrier_date"]
        - orders_clean["order_purchase_timestamp"]
    )
    .dt.total_seconds()
    / 86400
)

orders_clean["delivery_time_days"] = (
    (
        orders_clean["order_delivered_customer_date"]
        - orders_clean["order_purchase_timestamp"]
    )
    .dt.total_seconds()
    / 86400
)

orders_clean["delivery_delay_days"] = (
    (
        orders_clean["order_delivered_customer_date"]
        - orders_clean["order_estimated_delivery_date"]
    )
    .dt.total_seconds()
    / 86400
)

In [565]:
orders_clean[
    [
       "order_purchase_timestamp", 
        "order_approved_at", 
        "order_delivered_carrier_date", 
        "order_delivered_customer_date", 
        "order_estimated_delivery_date",
        "approval_time_hours", 
        "carrier_wait_days",
        "shipping_time_days",
        "delivery_time_days",
        "delivery_delay_days" 
    ]
].head(20)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_time_hours,carrier_wait_days,shipping_time_days,delivery_time_days,delivery_delay_days
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0.178333,2.366493,2.373924,8.436574,-7.107488
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,30.713889,0.462882,1.742627,13.782037,-5.355729
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0.276111,0.204595,0.216100,9.394213,-17.245498
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0.298056,3.745833,3.758252,13.208750,-12.980069
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.030556,0.893113,0.936053,2.873877,-9.238171
5,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,0.218889,1.699896,1.709016,16.542245,-5.543113
6,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,49.052500,NaN,NaN,NaN,NaN
7,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,0.194722,5.864988,5.873102,9.989826,-11.461215
8,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,32.360556,1.476204,2.824560,9.818762,-31.410995
9,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,0.175000,12.319352,12.326644,18.221852,-6.281597


## Status dostave prema kalendarskom datumu

Status se određuje usporedbom stvarnog i procijenjenog kalendarskog datuma. Ako stvarni datum dostave ne postoji, status ostaje nedostajuća vrijednost.

In [566]:
stvarni_datum_dostave = (
    orders_clean["order_delivered_customer_date"]
    .dt.normalize()
)
procijenjeni_datum_dostave = (
    orders_clean["order_estimated_delivery_date"]
    .dt.normalize()
)

orders_clean["delivery_status"] = pd.Series(
    pd.NA,
    index=orders_clean.index,
    dtype="string"
)

maska_status = (
    stvarni_datum_dostave.notna()
    & procijenjeni_datum_dostave.notna()
)

orders_clean.loc[
    maska_status
    & (stvarni_datum_dostave < procijenjeni_datum_dostave),
    "delivery_status"
] = "early"

orders_clean.loc[
    maska_status
    & (stvarni_datum_dostave == procijenjeni_datum_dostave),
    "delivery_status"
] = "on_time"

orders_clean.loc[
    maska_status
    & (stvarni_datum_dostave > procijenjeni_datum_dostave),
    "delivery_status"
] = "late"

orders_clean["delivered_within_deadline"] = pd.Series(
    pd.NA,
    index=orders_clean.index,
    dtype="boolean"
)

orders_clean.loc[
    orders_clean["delivery_status"].isin(["early", "on_time"]),
    "delivered_within_deadline"
] = True

orders_clean.loc[
    orders_clean["delivery_status"].eq("late"),
    "delivered_within_deadline"
] = False

In [567]:
zavrsna_provjera_orders = pd.Series({
    "broj_redaka": len(orders_clean),
    "duplicirani_order_id":
        orders_clean["order_id"].duplicated().sum(),
    "neocekivani_statusi":
        len(neocekivani_statusi),
    "nepovezani_kupci":
        (~orders_clean["customer_reference_valid"]).sum(),
    "kronoloske_anomalije":
        orders_clean["has_timeline_anomaly"].sum(),
    "status_dostave_nedostaje":
        orders_clean["delivery_status"].isna().sum()
})

zavrsna_provjera_orders

broj_redaka                 99441
duplicirani_order_id            0
neocekivani_statusi             0
nepovezani_kupci                0
kronoloske_anomalije         1382
status_dostave_nedostaje     2965
dtype: int64

### Zaključak čišćenja tablice `orders`

Vremenski atributi pretvoreni su u odgovarajući tip, statusi su standardizirani, a kronološke nekonzistentnosti zadržane su i označene. Izračunani su izvedeni atributi trajanja i statusa dostave.

# 6. Čišćenje tablice `order_items`

Čišćenje obuhvaća pretvorbu datuma, provjeru ključeva i rednih brojeva stavki, provjeru iznosa, izračun izvedenih financijskih atributa te označavanje statističkih outliera. Nijedan zapis ne uklanja se samo zato što je statistički neuobičajen.

In [568]:
print("Broj redaka:", len(order_items_clean))
print(
    "Broj potpuno dupliciranih redaka:",
    order_items_clean.duplicated().sum()
)
print(
    "Broj duplikata prema (order_id, order_item_id):",
    order_items_clean.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)
print("\nNedostajuće vrijednosti:")
print(order_items_clean.isna().sum())

Broj redaka: 112650
Broj potpuno dupliciranih redaka: 0
Broj duplikata prema (order_id, order_item_id): 0

Nedostajuće vrijednosti:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64


In [569]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)

## Kontinuitet rednih brojeva i referencijalni integritet

In [570]:
def redni_brojevi_su_kontinuirani(vrijednosti):
    stvarni = sorted(vrijednosti.astype(int).tolist())
    ocekivani = list(range(1, len(stvarni) + 1))
    return stvarni == ocekivani

kontinuitet_stavki = (
    order_items_clean
    .groupby("order_id")["order_item_id"]
    .apply(redni_brojevi_su_kontinuirani)
)

order_items_orders = order_items_clean.merge(
    orders_clean[["order_id"]],
    on="order_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)
order_items_products = order_items_clean.merge(
    products_clean[["product_id"]],
    on="product_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)
order_items_sellers = order_items_clean.merge(
    sellers_clean[["seller_id"]],
    on="seller_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

order_items_clean["order_reference_valid"] = (
    order_items_orders["_merge"].eq("both").to_numpy()
)
order_items_clean["product_reference_valid"] = (
    order_items_products["_merge"].eq("both").to_numpy()
)
order_items_clean["seller_reference_valid"] = (
    order_items_sellers["_merge"].eq("both").to_numpy()
)

## Financijski atributi i indikatori kvalitete

In [571]:
order_items_clean["price_invalid"] = (
    order_items_clean["price"].le(0)
)
order_items_clean["freight_invalid"] = (
    order_items_clean["freight_value"].lt(0)
)
order_items_clean["free_freight"] = (
    order_items_clean["freight_value"].eq(0)
)

order_items_clean["item_total_value"] = (
    order_items_clean["price"]
    + order_items_clean["freight_value"]
).round(2)

order_items_clean["freight_percentage"] = (
    order_items_clean["freight_value"]
    / order_items_clean["item_total_value"]
).round(4)

## Označavanje outliera cijene i dostave

In [572]:
q1_price = order_items_clean["price"].quantile(0.25)
q3_price = order_items_clean["price"].quantile(0.75)
iqr_price = q3_price - q1_price
lower_price = q1_price - 1.5 * iqr_price
upper_price = q3_price + 1.5 * iqr_price

order_items_clean["price_outlier"] = (
    (order_items_clean["price"] < lower_price)
    | (order_items_clean["price"] > upper_price)
)

q1_freight = order_items_clean["freight_value"].quantile(0.25)
q3_freight = order_items_clean["freight_value"].quantile(0.75)
iqr_freight = q3_freight - q1_freight
lower_freight = q1_freight - 1.5 * iqr_freight
upper_freight = q3_freight + 1.5 * iqr_freight

order_items_clean["freight_outlier"] = (
    (order_items_clean["freight_value"] < lower_freight)
    | (order_items_clean["freight_value"] > upper_freight)
)

order_items_clean["freight_greater_than_price"] = (
    order_items_clean["freight_value"]
    > order_items_clean["price"]
)

In [573]:
iznosi_po_narudzbi = (
    order_items_clean
    .groupby("order_id", as_index=False)
    .agg(
        ukupna_cijena=("price", "sum"),
        ukupna_dostava=("freight_value", "sum"),
        ukupna_vrijednost_stavki=("item_total_value", "sum"),
        broj_stavki=("order_item_id", "count")
    )
)

iznosi_po_narudzbi[
    [
        "ukupna_cijena",
        "ukupna_dostava",
        "ukupna_vrijednost_stavki"
    ]
] = iznosi_po_narudzbi[
    [
        "ukupna_cijena",
        "ukupna_dostava",
        "ukupna_vrijednost_stavki"
    ]
].round(2)

In [574]:
zavrsna_provjera_order_items = pd.Series({
    "broj_redaka": len(order_items_clean),
    "potpuni_duplikati":
        order_items_clean.duplicated().sum(),
    "duplikati_kljuca":
        order_items_clean.duplicated(
            subset=["order_id", "order_item_id"]
        ).sum(),
    "neispravne_sekvence":
        (~kontinuitet_stavki).sum(),
    "nepovezane_narudzbe":
        (~order_items_clean["order_reference_valid"]).sum(),
    "nepovezani_proizvodi":
        (~order_items_clean["product_reference_valid"]).sum(),
    "nepovezani_prodavaci":
        (~order_items_clean["seller_reference_valid"]).sum(),
    "neispravne_cijene":
        order_items_clean["price_invalid"].sum(),
    "negativna_dostava":
        order_items_clean["freight_invalid"].sum(),
    "besplatna_dostava":
        order_items_clean["free_freight"].sum(),
    "outlieri_cijene":
        order_items_clean["price_outlier"].sum(),
    "outlieri_dostave":
        order_items_clean["freight_outlier"].sum()
})

zavrsna_provjera_order_items

broj_redaka             112650
potpuni_duplikati            0
duplikati_kljuca             0
neispravne_sekvence          0
nepovezane_narudzbe          0
nepovezani_proizvodi         0
nepovezani_prodavaci         0
neispravne_cijene            0
negativna_dostava            0
besplatna_dostava          383
outlieri_cijene           8427
outlieri_dostave         12134
dtype: int64

### Zaključak čišćenja tablice `order_items`

Ključevi, datumi i iznosi provjereni su, izvedeni financijski atributi su dodani, a ekstremne vrijednosti zadržane su uz indikatore. Nijedan zapis nije uklonjen.

# 7. Čišćenje tablice `order_payments`

Čišćenje obuhvaća standardizaciju tipa plaćanja, provjeru ključeva i rednih brojeva plaćanja, označavanje neobičnih vrijednosti te usporedbu ukupno plaćenog iznosa s vrijednošću narudžbe.

Uočena odstupanja ne uklanjaju se. Zadržavaju se kao evidentirane anomalije koje se mogu dodatno razmatrati u tekstu rada.

In [575]:
print("Broj redaka:", len(payments_clean))
print(
    "Broj potpuno dupliciranih redaka:",
    payments_clean.duplicated().sum()
)
print(
    "Broj duplikata prema (order_id, payment_sequential):",
    payments_clean.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)
print("\nNedostajuće vrijednosti:")
print(payments_clean.isna().sum())

Broj redaka: 103886
Broj potpuno dupliciranih redaka: 0
Broj duplikata prema (order_id, payment_sequential): 0

Nedostajuće vrijednosti:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


## Standardizacija tipa plaćanja

In [576]:
payments_clean["payment_type"] = (
    payments_clean["payment_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

dozvoljeni_tipovi = {
    "credit_card",
    "debit_card",
    "voucher",
    "boleto",
    "not_defined"
}

neocekivani_tipovi = (
    set(payments_clean["payment_type"].dropna())
    - dozvoljeni_tipovi
)

assert len(neocekivani_tipovi) == 0

## Kontinuitet rednog broja plaćanja

Sekvence koje ne počinju brojem 1 ili imaju praznine ne ispravljaju se jer nije moguće pouzdano rekonstruirati nedostajuće transakcijske zapise. Takve narudžbe samo se označavaju.

In [577]:
sekvence_placanja = (
    payments_clean
    .groupby("order_id")["payment_sequential"]
    .apply(lambda x: sorted(x.astype(int).tolist()))
)

ispravna_sekvenca_placanja = (
    sekvence_placanja.apply(
        lambda x: x == list(range(1, len(x) + 1))
    )
)

order_ids_neispravne_sekvence = set(
    ispravna_sekvenca_placanja[
        ~ispravna_sekvenca_placanja
    ].index
)

payments_clean["payment_sequence_anomaly"] = (
    payments_clean["order_id"]
    .isin(order_ids_neispravne_sekvence)
)

## Označavanje neuobičajenih vrijednosti rata i iznosa

Nulte ili neuobičajene vrijednosti ne brišu se. Dodaju se indikatori kako bi se mogle jednostavno izdvojiti u kasnijim analizama.

In [578]:
payments_clean["installments_nonpositive"] = (
    payments_clean["payment_installments"].le(0)
)

payments_clean["payment_value_zero"] = (
    payments_clean["payment_value"].eq(0)
)

payments_clean["payment_value_negative"] = (
    payments_clean["payment_value"].lt(0)
)

## Referencijalni integritet prema narudžbama

In [579]:
payments_orders = payments_clean.merge(
    orders_clean[["order_id"]],
    on="order_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

payments_clean["order_reference_valid"] = (
    payments_orders["_merge"]
    .eq("both")
    .to_numpy()
)

## Označavanje statističkih outliera iznosa plaćanja

IQR metoda koristi se samo za indikator; vrijednosti se ne uklanjaju.

In [580]:
q1_payment = payments_clean["payment_value"].quantile(0.25)
q3_payment = payments_clean["payment_value"].quantile(0.75)
iqr_payment = q3_payment - q1_payment
lower_payment = q1_payment - 1.5 * iqr_payment
upper_payment = q3_payment + 1.5 * iqr_payment

payments_clean["payment_value_outlier"] = (
    (payments_clean["payment_value"] < lower_payment)
    | (payments_clean["payment_value"] > upper_payment)
)

## Usporedba plaćenog i obračunatog iznosa

Ukupni iznos plaćanja uspoređuje se sa zbrojem cijene proizvoda i dostave. Razlike se ne uklanjaju niti automatski ispravljaju, u skladu s odlukom da skladište zadrži evidentirane anomalije.

In [581]:
payments_total = (
    payments_clean
    .groupby("order_id", as_index=False)
    .agg(
        ukupno_placeno=("payment_value", "sum"),
        broj_payment_zapisa=("payment_sequential", "count")
    )
)

order_items_total = (
    order_items_clean
    .groupby("order_id", as_index=False)
    .agg(
        ukupna_cijena_proizvoda=("price", "sum"),
        ukupna_dostava=("freight_value", "sum")
    )
)

order_items_total["ocekivani_iznos"] = (
    order_items_total["ukupna_cijena_proizvoda"]
    + order_items_total["ukupna_dostava"]
)

payments_vs_orders = payments_total.merge(
    order_items_total,
    on="order_id",
    how="outer",
    validate="one_to_one",
    indicator=True
)

novcani_stupci = [
    "ukupno_placeno",
    "ukupna_cijena_proizvoda",
    "ukupna_dostava",
    "ocekivani_iznos"
]

payments_vs_orders[novcani_stupci] = (
    payments_vs_orders[novcani_stupci]
    .round(2)
)

payments_vs_orders["razlika_placanje_narudzba"] = (
    payments_vs_orders["ukupno_placeno"]
    - payments_vs_orders["ocekivani_iznos"]
).round(2)

payments_vs_orders["payment_amount_mismatch"] = (
    payments_vs_orders[
        "razlika_placanje_narudzba"
    ].abs().ge(0.01)
)

mismatch_order_ids = set(
    payments_vs_orders.loc[
        payments_vs_orders["payment_amount_mismatch"],
        "order_id"
    ]
)

payments_clean["payment_amount_mismatch"] = (
    payments_clean["order_id"]
    .isin(mismatch_order_ids)
)

In [582]:
zavrsna_provjera_payments = pd.Series({
    "broj_redaka": len(payments_clean),
    "potpuni_duplikati":
        payments_clean.duplicated().sum(),
    "duplikati_kljuca":
        payments_clean.duplicated(
            subset=["order_id", "payment_sequential"]
        ).sum(),
    "neocekivani_tipovi":
        len(neocekivani_tipovi),
    "neispravne_sekvence_narudzbi":
        len(order_ids_neispravne_sekvence),
    "rate_manje_ili_jednake_nuli":
        payments_clean["installments_nonpositive"].sum(),
    "placanje_jednako_nuli":
        payments_clean["payment_value_zero"].sum(),
    "negativno_placanje":
        payments_clean["payment_value_negative"].sum(),
    "nepovezane_narudzbe":
        (~payments_clean["order_reference_valid"]).sum(),
    "outlieri_placanja":
        payments_clean["payment_value_outlier"].sum(),
    "narudzbe_s_razlikom_iznosa":
        payments_vs_orders[
            "payment_amount_mismatch"
        ].sum()
})

zavrsna_provjera_payments

broj_redaka                     103886
potpuni_duplikati                    0
duplikati_kljuca                     0
neocekivani_tipovi                   0
neispravne_sekvence_narudzbi        80
rate_manje_ili_jednake_nuli          2
placanje_jednako_nuli                9
negativno_placanje                   0
nepovezane_narudzbe                  0
outlieri_placanja                 7981
narudzbe_s_razlikom_iznosa         576
dtype: int64

### Zaključak čišćenja tablice `order_payments`

Tipovi plaćanja standardizirani su, a referencijalni integritet provjeren. Neuobičajene sekvence, nepozitivne rate, nulta plaćanja, statistički outlieri i razlike između plaćenog i obračunatog iznosa zadržani su te označeni odgovarajućim indikatorima. Nijedan zapis nije uklonjen.

# 8. Čišćenje tablice `order_reviews`

Tablica `order_reviews` sadrži ocjene i tekstualne komentare korisnika.

Jedna narudžba može biti povezana s više zapisa recenzija, a isti `review_id` može se pojaviti u više redaka. Takvi zapisi ne smatraju se automatski duplikatima jer mogu predstavljati naknadno ažurirane ili ponovno zabilježene recenzije.

Za analize na razini narudžbe zadržava se posljednja dostupna recenzija prema vremenskom slijedu. Na taj način svaka narudžba sudjeluje najviše jednom, a zadržava se posljednja poznata ocjena korisnika.

Tekstualni naslov i komentar nisu obavezni te njihov izostanak ne predstavlja pogrešku kvalitete podataka.

In [583]:
print("Broj redaka:", len(reviews_clean))

print(
    "Broj potpuno dupliciranih redaka:",
    reviews_clean.duplicated().sum()
)

print(
    "Broj jedinstvenih review_id:",
    reviews_clean["review_id"].nunique()
)

print(
    "Broj jedinstvenih order_id:",
    reviews_clean["order_id"].nunique()
)

print("\nNedostajuće vrijednosti:")
print(reviews_clean.isna().sum())

Broj redaka: 99224
Broj potpuno dupliciranih redaka: 0
Broj jedinstvenih review_id: 98410
Broj jedinstvenih order_id: 98673

Nedostajuće vrijednosti:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


## Pretvorba vremenskih atributa

Atributi vremena nastanka ankete i odgovora na nju pretvaraju se u odgovarajući vremenski tip.

Neuspješne pretvorbe ostale bi označene vrijednošću `NaT`.

In [584]:
datum_stupci_reviews = [
    "review_creation_date",
    "review_answer_timestamp"
]

for stupac in datum_stupci_reviews:
    reviews_clean[stupac] = pd.to_datetime(
        reviews_clean[stupac],
        errors="coerce"
    )

print(
    reviews_clean[datum_stupci_reviews]
    .isna()
    .sum()
)

review_creation_date       0
review_answer_timestamp    0
dtype: int64


## Provjera valjanosti ocjena

Ocjena recenzije treba biti cijeli broj od 1 do 5.

Zapisi izvan tog raspona ne bi se automatski brisali, nego bi bili označeni kao nevaljani.

In [585]:
reviews_clean["review_score_invalid"] = (
    reviews_clean["review_score"].isna()
    |
    ~reviews_clean["review_score"].between(
        1,
        5,
        inclusive="both"
    )
)

print(
    "Broj recenzija s nevaljanom ocjenom:",
    reviews_clean["review_score_invalid"].sum()
)

print("\nRaspodjela ocjena:")
print(
    reviews_clean["review_score"]
    .value_counts(dropna=False)
    .sort_index()
)

Broj recenzija s nevaljanom ocjenom: 0

Raspodjela ocjena:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


## Provjera referencijalnog integriteta

Provjerava se postoji li svaka narudžba iz tablice `order_reviews` u očišćenoj tablici `orders`.

In [586]:
reviews_orders = reviews_clean.merge(
    orders_clean[["order_id"]],
    on="order_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

reviews_clean["order_reference_valid"] = (
    reviews_orders["_merge"]
    .eq("both")
    .to_numpy()
)

print(
    "Broj recenzija bez postojeće narudžbe:",
    (~reviews_clean["order_reference_valid"]).sum()
)

Broj recenzija bez postojeće narudžbe: 0


## Označavanje tekstualnih sadržaja

Nedostajući naslov ili komentar ne smatra se pogreškom jer su oba atributa neobavezna.

Dodaju se indikatori koji označavaju postoji li naslov i komentar recenzije.

In [587]:
reviews_clean["has_review_title"] = (
    reviews_clean["review_comment_title"]
    .notna()
    &
    reviews_clean["review_comment_title"]
    .astype("string")
    .str.strip()
    .ne("")
)

reviews_clean["has_review_message"] = (
    reviews_clean["review_comment_message"]
    .notna()
    &
    reviews_clean["review_comment_message"]
    .astype("string")
    .str.strip()
    .ne("")
)

print(
    "Broj recenzija s naslovom:",
    reviews_clean["has_review_title"].sum()
)

print(
    "Broj recenzija s tekstualnim komentarom:",
    reviews_clean["has_review_message"].sum()
)

Broj recenzija s naslovom: 11566
Broj recenzija s tekstualnim komentarom: 40950


## Identifikacija narudžbi s više zapisa recenzija

Prije izdvajanja posljednje recenzije označava se koliko zapisa recenzija pripada svakoj narudžbi.

In [588]:
broj_recenzija_po_narudzbi = (
    reviews_clean
    .groupby("order_id")
    .size()
    .rename("review_records_count")
)

reviews_clean = reviews_clean.merge(
    broj_recenzija_po_narudzbi,
    on="order_id",
    how="left",
    validate="many_to_one"
)

reviews_clean["multiple_reviews_for_order"] = (
    reviews_clean["review_records_count"] > 1
)

print(
    "Broj narudžbi s više zapisa recenzija:",
    (
        broj_recenzija_po_narudzbi > 1
    ).sum()
)

print(
    "Broj redaka koji pripadaju narudžbama s više recenzija:",
    reviews_clean["multiple_reviews_for_order"].sum()
)

Broj narudžbi s više zapisa recenzija: 547
Broj redaka koji pripadaju narudžbama s više recenzija: 1098


In [589]:
# Vrijeme odgovora za svaki pojedinačni zapis recenzije
reviews_clean["review_response_time_hours"] = (
    (
        reviews_clean["review_answer_timestamp"]
        - reviews_clean["review_creation_date"]
    )
    .dt.total_seconds()
    / 3600
)

reviews_clean["review_response_before_creation"] = (
    reviews_clean["review_response_time_hours"] < 0
)

# Prvi odgovor za svaki par (narudžba, datum stvaranja ankete)
prvi_odgovori = (
    reviews_clean
    .loc[
        reviews_clean["review_response_time_hours"].ge(0)
    ]
    .groupby(
        ["order_id", "review_creation_date"],
        as_index=False
    )
    .agg(
        first_review_answer_timestamp=(
            "review_answer_timestamp", "min"
        )
    )
)

prvi_odgovori["first_review_response_time_hours"] = (
    (
        prvi_odgovori["first_review_answer_timestamp"]
        - prvi_odgovori["review_creation_date"]
    )
    .dt.total_seconds()
    / 3600
)

display(prvi_odgovori.head())

,order_id,review_creation_date,first_review_answer_timestamp,first_review_response_time_hours
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-21,2017-09-22 10:57:03,34.950833
1,00018f77f2f0320c557190d7a144bdd3,2017-05-13,2017-05-15 11:34:13,59.570278
2,000229ec398224ef6ca0657da4fc703e,2018-01-23,2018-01-23 16:06:31,16.108611
3,00024acbcdf0a6daa1e931b038114c75,2018-08-15,2018-08-15 16:39:01,16.650278
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-03-02,2017-03-03 10:54:59,34.916389


## Odabir posljednje dostupne recenzije po narudžbi

Za određivanje posljednje recenzije koristi se prvenstveno `review_answer_timestamp`, a zatim `review_creation_date`.

Ako više zapisa ima isti vremenski trenutak, kao dodatni stabilni kriterij koristi se izvorni redoslijed zapisa u tablici.

Rezultat se sprema u novu tablicu `reviews_latest_clean`, dok `reviews_clean` zadržava sve izvorne zapise i dodane indikatore kvalitete.

In [590]:
reviews_clean["_review_source_order"] = np.arange(
    len(reviews_clean)
)

reviews_latest_clean = (
    reviews_clean
    .sort_values(
        [
            "order_id",
            "review_answer_timestamp",
            "review_creation_date",
            "_review_source_order"
        ],
        na_position="first"
    )
    .drop_duplicates(
        subset=["order_id"],
        keep="last"
    )
    .drop(
        columns=["_review_source_order"]
    )
    .reset_index(drop=True)
)

reviews_clean = reviews_clean.drop(
    columns=["_review_source_order"]
)

## Provjera rezultata izdvajanja posljednje recenzije

Tablica `reviews_latest_clean` treba sadržavati najviše jedan zapis po narudžbi.

In [591]:
print(
    "Broj redaka u izvornom očišćenom skupu:",
    len(reviews_clean)
)

print(
    "Broj redaka nakon odabira posljednje recenzije:",
    len(reviews_latest_clean)
)

print(
    "Broj dupliciranih order_id u reviews_latest_clean:",
    reviews_latest_clean["order_id"]
    .duplicated()
    .sum()
)

assert (
    reviews_latest_clean["order_id"]
    .duplicated()
    .sum()
    == 0
)

Broj redaka u izvornom očišćenom skupu: 99224
Broj redaka nakon odabira posljednje recenzije: 98673
Broj dupliciranih order_id u reviews_latest_clean: 0


In [592]:
reviews_latest_clean = reviews_latest_clean.merge(
    prvi_odgovori[
        [
            "order_id",
            "review_creation_date",
            "first_review_answer_timestamp",
            "first_review_response_time_hours"
        ]
    ],
    on=["order_id", "review_creation_date"],
    how="left",
    validate="many_to_one"
)

reviews_latest_clean["review_response_time_hours"] = (
    (
        reviews_latest_clean["review_answer_timestamp"]
        - reviews_latest_clean["review_creation_date"]
    )
    .dt.total_seconds()
    / 3600
)

## Završna provjera tablice `order_reviews`

In [593]:
zavrsna_provjera_reviews = pd.Series({
    "broj_izvornih_redaka":
        len(reviews_clean),

    "broj_posljednjih_recenzija":
        len(reviews_latest_clean),

    "potpuni_duplikati":
        reviews_clean.duplicated().sum(),

    "nevaljane_ocjene":
        reviews_clean["review_score_invalid"].sum(),

    "recenzije_bez_narudzbe":
        (~reviews_clean["order_reference_valid"]).sum(),

    "narudzbe_s_vise_recenzija":
        (
            broj_recenzija_po_narudzbi > 1
        ).sum(),

    "duplicirani_order_id_nakon_odabira":
        reviews_latest_clean[
            "order_id"
        ].duplicated().sum(),

    "odgovor_prije_nastanka_recenzije":
        reviews_clean[
            "review_response_before_creation"
        ].sum(),

    "recenzije_s_naslovom":
        reviews_clean["has_review_title"].sum(),

    "recenzije_s_komentarom":
        reviews_clean["has_review_message"].sum()
})

zavrsna_provjera_reviews

broj_izvornih_redaka                  99224
broj_posljednjih_recenzija            98673
potpuni_duplikati                         0
nevaljane_ocjene                          0
recenzije_bez_narudzbe                    0
narudzbe_s_vise_recenzija               547
duplicirani_order_id_nakon_odabira        0
odgovor_prije_nastanka_recenzije          0
recenzije_s_naslovom                  11566
recenzije_s_komentarom                40950
dtype: int64

### Zaključak čišćenja tablice `order_reviews`

Izvorni zapisi recenzija nisu uklonjeni iz tablice `reviews_clean`.

Nedostajući naslovi i tekstualni komentari zadržani su jer predstavljaju neobavezne atribute, a ne pogreške kvalitete podataka. Ocjene, vremenski atributi i referencijalni integritet provjereni su te su dodani odgovarajući indikatori kvalitete.

Za analize na razini narudžbe izrađena je tablica `reviews_latest_clean`, koja sadrži posljednju dostupnu recenziju za svaku narudžbu. Time svaka narudžba sudjeluje najviše jednom, a zadržava se posljednja poznata ocjena korisnika.

# 9. Završna provjera tablice `category_translation`

Referentna tablica prijevoda već je dopunjena tijekom čišćenja tablice `products`.

U radnu kopiju prethodno su dodani:

- prijevod kategorije `pc_gamer`;
- prijevod kategorije `portateis_cozinha_e_preparadores_de_alimentos`;
- zapis `unknown` za proizvode bez izvorno definirane kategorije.

U ovom dijelu ne provodi se ponovno dopunjavanje tablice, nego se samo provjeravaju njezina potpunost, jedinstvenost i povezanost s očišćenim kategorijama proizvoda.

## Osnovna provjera referentne tablice

In [594]:
print("Broj redaka:", len(category_translation_clean))

print(
    "Broj potpuno dupliciranih redaka:",
    category_translation_clean.duplicated().sum()
)

print(
    "Broj dupliciranih portugalskih kategorija:",
    category_translation_clean[
        "product_category_name"
    ].duplicated().sum()
)

print("\nNedostajuće vrijednosti:")
print(category_translation_clean.isna().sum())

Broj redaka: 74
Broj potpuno dupliciranih redaka: 0
Broj dupliciranih portugalskih kategorija: 0

Nedostajuće vrijednosti:
product_category_name            0
product_category_name_english    0
dtype: int64


## Provjera jedinstvenosti i obaveznih dopuna

Svaka portugalska kategorija treba imati najviše jedan engleski prijevod. Također se provjerava postoje li sva tri zapisa dodana tijekom čišćenja tablice `products`.

In [595]:
duplicirane_kategorije = (
    category_translation_clean[
        "product_category_name"
    ]
    .duplicated()
    .sum()
)

obavezne_dopune = {
    "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos",
    "unknown"
}

postojece_kategorije = set(
    category_translation_clean[
        "product_category_name"
    ].dropna()
)

nedostajuce_dopune = (
    obavezne_dopune
    - postojece_kategorije
)

print(
    "Broj dupliciranih kategorija:",
    duplicirane_kategorije
)

print(
    "Nedostajuće obavezne dopune:",
    sorted(nedostajuce_dopune)
)

assert duplicirane_kategorije == 0
assert len(nedostajuce_dopune) == 0

Broj dupliciranih kategorija: 0
Nedostajuće obavezne dopune: []


## Provjera povezanosti s očišćenim kategorijama proizvoda

Provjera se provodi preko atributa `product_category_name_clean`.

Taj atribut sadrži vrijednost `unknown` za 610 proizvoda kojima u izvornom skupu nedostaje naziv kategorije. Zbog toga se ne koristi izvorni atribut `product_category_name`, u kojem su te vrijednosti i dalje namjerno sačuvane kao nedostajuće.

In [596]:
translation_lookup = (
    category_translation_clean
    .set_index("product_category_name")[
        "product_category_name_english"
    ]
)

products_translation_check = (
    products_clean[
        [
            "product_id",
            "product_category_name",
            "product_category_name_clean"
        ]
    ]
    .copy()
)

products_translation_check[
    "translation_check"
] = (
    products_translation_check[
        "product_category_name_clean"
    ]
    .map(translation_lookup)
)

products_translation_check[
    "translation_available"
] = (
    products_translation_check[
        "translation_check"
    ].notna()
)

print(
    "Broj proizvoda s povezanim prijevodom:",
    products_translation_check[
        "translation_available"
    ].sum()
)

print(
    "Broj proizvoda bez povezanog prijevoda:",
    (
        ~products_translation_check[
            "translation_available"
        ]
    ).sum()
)

Broj proizvoda s povezanim prijevodom: 32951
Broj proizvoda bez povezanog prijevoda: 0


## Posebna provjera proizvoda s kategorijom `unknown`

Proizvodi bez izvorno definirane kategorije trebaju biti povezani sa zapisom `unknown` i engleskim nazivom `Unknown category`.

In [597]:
unknown_products_check = (
    products_translation_check[
        products_translation_check[
            "product_category_name_clean"
        ].eq("unknown")
    ]
    .copy()
)

print(
    "Broj proizvoda s očišćenom kategorijom unknown:",
    len(unknown_products_check)
)

print(
    "Broj proizvoda unknown bez prijevoda:",
    unknown_products_check[
        "translation_check"
    ].isna().sum()
)

print(
    "Engleske vrijednosti povezane s unknown:",
    sorted(
        unknown_products_check[
            "translation_check"
        ]
        .dropna()
        .unique()
        .tolist()
    )
)

Broj proizvoda s očišćenom kategorijom unknown: 610
Broj proizvoda unknown bez prijevoda: 0
Engleske vrijednosti povezane s unknown: ['Unknown category']


## Završna provjera tablice `category_translation`

In [598]:
zavrsna_provjera_translation = pd.Series({
    "broj_redaka":
        len(category_translation_clean),

    "potpuni_duplikati":
        category_translation_clean
        .duplicated()
        .sum(),

    "duplicirane_kategorije":
        category_translation_clean[
            "product_category_name"
        ]
        .duplicated()
        .sum(),

    "nedostajuce_vrijednosti":
        category_translation_clean
        .isna()
        .sum()
        .sum(),

    "nedostajuce_obavezne_dopune":
        len(nedostajuce_dopune),

    "proizvodi_bez_prijevoda":
        (
            ~products_translation_check[
                "translation_available"
            ]
        ).sum(),

    "unknown_proizvodi_bez_prijevoda":
        unknown_products_check[
            "translation_check"
        ]
        .isna()
        .sum()
})

zavrsna_provjera_translation

broj_redaka                        74
potpuni_duplikati                   0
duplicirane_kategorije              0
nedostajuce_vrijednosti             0
nedostajuce_obavezne_dopune         0
proizvodi_bez_prijevoda             0
unknown_proizvodi_bez_prijevoda     0
dtype: int64

### Zaključak završne provjere tablice `category_translation`

Dopunjavanje referentne tablice provedeno je ranije tijekom čišćenja tablice `products` i ovdje nije ponavljano.

Provjera se temelji na očišćenom atributu `product_category_name_clean`, zbog čega su i proizvodi bez izvorne kategorije ispravno povezani sa zapisom `unknown`.

Referentna tablica sada sadrži jedinstven prijevod za svaku očišćenu kategoriju proizvoda.

# 10. Završna validacija očišćenih podataka

Prije spremanja očišćenih skupova provode se završne tehničke provjere.

Cilj je potvrditi:

- jedinstvenost primarnih i složenih ključeva;
- očuvanje broja redaka ondje gdje zapisi nisu uklanjani;
- referencijalni integritet između povezanih tablica;
- potpunost prijevoda kategorija;
- najviše jednu odabranu recenziju po narudžbi.

Prepoznate anomalije koje se prema odluci ne uklanjaju ostaju sačuvane kroz odgovarajuće indikatore kvalitete.

In [599]:
zavrsna_validacija = pd.Series({
    "customers_customer_id_unique":
        customers_clean["customer_id"].is_unique,

    "sellers_seller_id_unique":
        sellers_clean["seller_id"].is_unique,

    "products_product_id_unique":
        products_clean["product_id"].is_unique,

    "orders_order_id_unique":
        orders_clean["order_id"].is_unique,

    "order_items_key_unique":
        ~order_items_clean.duplicated(
            subset=["order_id", "order_item_id"]
        ).any(),

    "payments_key_unique":
        ~payments_clean.duplicated(
            subset=["order_id", "payment_sequential"]
        ).any(),

    "reviews_latest_order_id_unique":
        reviews_latest_clean["order_id"].is_unique,

    "translation_category_unique":
        category_translation_clean[
            "product_category_name"
        ].is_unique,

    "products_without_translation":
        (
            ~products_translation_check[
                "translation_available"
            ]
        ).sum()
        == 0,

    "order_items_orders_valid":
        order_items_clean[
            "order_reference_valid"
        ].all(),

    "order_items_products_valid":
        order_items_clean[
            "product_reference_valid"
        ].all(),

    "order_items_sellers_valid":
        order_items_clean[
            "seller_reference_valid"
        ].all(),

    "payments_orders_valid":
        payments_clean[
            "order_reference_valid"
        ].all(),

    "reviews_orders_valid":
        reviews_clean[
            "order_reference_valid"
        ].all()
})

zavrsna_validacija

customers_customer_id_unique      True
sellers_seller_id_unique          True
products_product_id_unique        True
orders_order_id_unique            True
order_items_key_unique            True
payments_key_unique               True
reviews_latest_order_id_unique    True
translation_category_unique       True
products_without_translation      True
order_items_orders_valid          True
order_items_products_valid        True
order_items_sellers_valid         True
payments_orders_valid             True
reviews_orders_valid              True
dtype: bool

In [600]:
neuspjele_provjere = (
    zavrsna_validacija[
        ~zavrsna_validacija.astype(bool)
    ]
)

print(
    "Broj neuspjelih završnih provjera:",
    len(neuspjele_provjere)
)

if len(neuspjele_provjere) > 0:
    print("\nNeuspjele provjere:")
    print(neuspjele_provjere)
else:
    print(
        "\n✓ Sve završne tehničke provjere "
        "uspješno su prošle."
    )

Broj neuspjelih završnih provjera: 0

✓ Sve završne tehničke provjere uspješno su prošle.


# 11. Spremanje očišćenih skupova podataka

Očišćene tablice spremaju se u zaseban direktorij kako bi izvorni CSV skupovi ostali nepromijenjeni.

Uz glavne očišćene tablice spremaju se:

- `reviews_latest_clean`, s posljednjom recenzijom po narudžbi;
- sažetak odstupanja plaćenog i obračunatog iznosa, radi dokumentiranja prepoznatih anomalija.

Indeks DataFramea ne sprema se u CSV datoteke.

In [601]:
# Direktorij za spremanje očišćenih podataka

CLEAN_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Direktorij za spremanje:")
print(CLEAN_DATA_DIR)

Direktorij za spremanje:
C:\Users\Ana\Desktop\diplomskiPrakticni\data\clean


In [602]:
# Izdvajanje narudžbi s razlikom između očekivanog i plaćenog iznosa

placanja_odstupanja = (
    payments_vs_orders[
        payments_vs_orders["payment_amount_mismatch"]
    ]
    .copy()
    .sort_values("order_id")
)

print(
    "Broj narudžbi s odstupanjem:",
    len(placanja_odstupanja)
)

placanja_odstupanja.head()

Broj narudžbi s odstupanjem: 576


,order_id,ukupno_placeno,broj_payment_zapisa,ukupna_cijena_proizvoda,ukupna_dostava,ocekivani_iznos,_merge,razlika_placanje_narudzba,payment_amount_mismatch
135,005d9a5423d47281ac463a968b3936fb,145.26,1.0,99.97,45.28,145.25,both,0.01,True
166,00789ce015e7e5791c7914f32bb4fad4,190.81,1.0,154.00,14.83,168.83,both,21.98,True
533,016726239765c18f66826453f39c64e3,265.77,1.0,219.99,15.14,235.13,both,30.64,True
733,01e51b7c3025655646143d09b911e1d7,35.02,1.0,19.00,14.10,33.10,both,1.92,True
974,028aa7c930356788f861ed1b7f984819,62.94,1.0,39.90,17.63,57.53,both,5.41,True


In [603]:
products_clean = products_clean.drop(
    columns=[
        "weight_low_outlier",
        "weight_high_outlier",
        "weight_outlier",
        "volume_low_outlier",
        "volume_high_outlier",
        "volume_outlier"
    ],
    errors="ignore"
)

In [604]:
# Spremanje očišćenih tablica

customers_clean.to_csv(
    CLEAN_DATA_DIR / "olist_customers_dataset_clean.csv",
    index=False
)

geolocation_clean.to_csv(
    CLEAN_DATA_DIR / "olist_geolocation_dataset_clean.csv",
    index=False
)

sellers_clean.to_csv(
    CLEAN_DATA_DIR / "olist_sellers_dataset_clean.csv",
    index=False
)

products_clean.to_csv(
    CLEAN_DATA_DIR / "olist_products_dataset_clean.csv",
    index=False
)

orders_clean.to_csv(
    CLEAN_DATA_DIR / "olist_orders_dataset_clean.csv",
    index=False
)

order_items_clean.to_csv(
    CLEAN_DATA_DIR / "olist_order_items_dataset_clean.csv",
    index=False
)

payments_clean.to_csv(
    CLEAN_DATA_DIR / "olist_order_payments_dataset_clean.csv",
    index=False
)

reviews_latest_clean.to_csv(
    CLEAN_DATA_DIR / "olist_order_reviews_dataset_clean.csv",
    index=False
)

category_translation_clean.to_csv(
    CLEAN_DATA_DIR / "product_category_name_translation_clean.csv",
    index=False
)

placanja_odstupanja.to_csv(
    CLEAN_DATA_DIR / "payment_amount_anomalies.csv",
    index=False
)

print("✓ Sve datoteke uspješno su spremljene.")

✓ Sve datoteke uspješno su spremljene.


In [605]:
ocekivane_datoteke = [
    "olist_customers_dataset_clean.csv",
    "olist_geolocation_dataset_clean.csv",
    "olist_sellers_dataset_clean.csv",
    "olist_products_dataset_clean.csv",
    "olist_orders_dataset_clean.csv",
    "olist_order_items_dataset_clean.csv",
    "olist_order_payments_dataset_clean.csv",
    "olist_order_reviews_dataset_clean.csv",
    "product_category_name_translation_clean.csv",
    "payment_amount_anomalies.csv"
]

print("Provjera spremljenih datoteka:\n")

for datoteka in ocekivane_datoteke:

    postoji = (
        CLEAN_DATA_DIR / datoteka
    ).exists()

    print(
        f"{datoteka:<45} {'✓' if postoji else '✗'}"
    )

assert all(
    (
        CLEAN_DATA_DIR / datoteka
    ).exists()
    for datoteka in ocekivane_datoteke
)

print("\nSve datoteke uspješno su spremljene.")

Provjera spremljenih datoteka:

olist_customers_dataset_clean.csv             ✓
olist_geolocation_dataset_clean.csv           ✓
olist_sellers_dataset_clean.csv               ✓
olist_products_dataset_clean.csv              ✓
olist_orders_dataset_clean.csv                ✓
olist_order_items_dataset_clean.csv           ✓
olist_order_payments_dataset_clean.csv        ✓
olist_order_reviews_dataset_clean.csv         ✓
product_category_name_translation_clean.csv   ✓
payment_amount_anomalies.csv                  ✓

Sve datoteke uspješno su spremljene.


In [606]:
print("=" * 70)
print("ČIŠĆENJE PODATAKA USPJEŠNO ZAVRŠENO")
print("=" * 70)

print()

print("Očišćene tablice spremljene su u:")
print(CLEAN_DATA_DIR)

print()

print("Za ETL proces koriste se sljedeće tablice:")

for naziv in ocekivane_datoteke:
    print(f" • {naziv}")

print()

print("Napomena:")
print("- anomalije plaćanja nisu uklonjene")
print("- spremljene su zasebno radi dokumentiranja")
print("- za recenzije je spremljena posljednja recenzija po narudžbi")

ČIŠĆENJE PODATAKA USPJEŠNO ZAVRŠENO

Očišćene tablice spremljene su u:
C:\Users\Ana\Desktop\diplomskiPrakticni\data\clean

Za ETL proces koriste se sljedeće tablice:
 • olist_customers_dataset_clean.csv
 • olist_geolocation_dataset_clean.csv
 • olist_sellers_dataset_clean.csv
 • olist_products_dataset_clean.csv
 • olist_orders_dataset_clean.csv
 • olist_order_items_dataset_clean.csv
 • olist_order_payments_dataset_clean.csv
 • olist_order_reviews_dataset_clean.csv
 • product_category_name_translation_clean.csv
 • payment_amount_anomalies.csv

Napomena:
- anomalije plaćanja nisu uklonjene
- spremljene su zasebno radi dokumentiranja
- za recenzije je spremljena posljednja recenzija po narudžbi


# Zaključak faze čišćenja podataka

Tijekom faze čišćenja provedene su provjere kvalitete podataka, standardizacija tekstualnih atributa, validacija referencijalnog integriteta, dopuna nedostajućih vrijednosti kada je to bilo moguće te izračun izvedenih atributa potrebnih za daljnju izgradnju skladišta podataka.

Anomalije koje nije bilo moguće pouzdano ispraviti nisu uklonjene iz skupa podataka. Umjesto toga označene su odgovarajućim indikatorima ili izdvojene u zasebne pomoćne tablice, čime je očuvana izvornost podataka uz potpunu dokumentiranost uočenih odstupanja.

Očišćeni skupovi podataka spremljeni su u direktorij `data/clean` i spremni su za daljnji ETL proces te izgradnju skladišta podataka u PostgreSQL sustavu.